# RSNA Knee — hierarchical MIL successor to verified 0.877

This notebook is the accuracy-first successor to the completed **0.877** submission. The
byte-identical run-2 notebook is archived in `10_experiments/`; this generated notebook does
not claim an improvement until a completed Kaggle run scores above 0.877.

The verified run-2 70/30 detail/depth ensemble is immutable and is always written first. One
new equal-fold DINOv2-S bank tests a controlled hierarchical-attention change:

| Bank | Input | Cached slices/slot | Windows | Training bag | Slice band | Seed |
|---|---:|---:|---:|---:|---:|---:|
| hierarchical | 224 px | 12 | 10 | 2 separated windows | 0.12–0.88 | 7301 |

The new head learns target-aware attention at patch, acquisition-slot, and ordered-window
levels. It retains the six physical slots, anatomical priors, the last six trainable encoder
blocks, 130 mm crop, physical laterality, and no horizontal flip. Only the matching run-2
depth backbone is loaded; the competition head is learned in this run.

Run modes fail closed:

- `RSNA_MODE=train` requires `RSNA_LABELS_PATH` and `RSNA_RUN2_PATH`.
- `RSNA_MODE=infer` requires `RSNA_WEIGHTS_DIR` and `RSNA_RUN2_PATH`.
- `RSNA_MODE=smoke` is non-production; debug labels additionally require
  `RSNA_ALLOW_LEXICON_DEBUG=1` and can never create `submission.csv`.


In [ ]:
# Kaggle launch configuration and fail-closed production preflight.
# Change only RSNA_MODE to "smoke" for a shakeout or "infer" for manifest-only inference.
import hashlib
import os
import zipfile
from pathlib import Path

import pandas as pd

os.environ.setdefault("RSNA_MODE", "train")
os.environ.setdefault(
    "RSNA_INPUT_DIR",
    "/kaggle/input/competitions/rsna-knee-abnormality-detection",
)
os.environ.setdefault(
    "RSNA_DINOV2_DIR",
    "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1",
)
os.environ.pop("RSNA_ALLOW_LEXICON_DEBUG", None)

RUN2_DATASET = Path(os.environ.get(
    "RSNA_RUN2_DATASET",
    "/kaggle/input/datasets/bikashsah1230/bikash-training-weight",
))
EXPECTED_LABEL_SHA256 = "13b0be74f9844c38181d096145fc39041e8e45c0b57243d83e2f5d01f29132c4"
UID = "StudyInstanceUID"
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

if not os.environ.get("RSNA_RUN2_PATH", "").strip():
    zip_candidates = sorted(RUN2_DATASET.rglob("results.zip"))
    complete_labels = sorted(RUN2_DATASET.rglob("report_labels_v2_complete.csv"))
    if zip_candidates:
        os.environ["RSNA_RUN2_PATH"] = str(zip_candidates[0])
    elif complete_labels:
        os.environ["RSNA_RUN2_PATH"] = str(complete_labels[0].parent)
    else:
        raise FileNotFoundError(
            f"No complete run-2 results.zip or extracted package under {RUN2_DATASET}")

run2_path = Path(os.environ["RSNA_RUN2_PATH"])
if not (run2_path.is_dir() or run2_path.is_file()):
    raise FileNotFoundError(f"RSNA_RUN2_PATH does not exist: {run2_path}")

mode = os.environ["RSNA_MODE"].strip().lower()
if mode != "infer":
    if not os.environ.get("RSNA_LABELS_PATH", "").strip():
        if run2_path.is_file() and run2_path.suffix.lower() == ".zip":
            label_path = Path("/kaggle/working/report_labels_v2_complete.csv")
            with zipfile.ZipFile(run2_path) as archive:
                label_path.write_bytes(archive.read("report_labels_v2_complete.csv"))
        else:
            label_candidates = sorted(run2_path.rglob("report_labels_v2_complete.csv"))
            if not label_candidates:
                raise FileNotFoundError(
                    f"report_labels_v2_complete.csv is missing below {run2_path}")
            label_path = label_candidates[0]
        os.environ["RSNA_LABELS_PATH"] = str(label_path)

    label_path = Path(os.environ["RSNA_LABELS_PATH"])
    train_path = Path(os.environ["RSNA_INPUT_DIR"]) / "train.csv"
    if not label_path.is_file() or not train_path.is_file():
        raise FileNotFoundError(f"missing label/train input: {label_path}, {train_path}")
    train = pd.read_csv(train_path, dtype={UID: str})
    labels = pd.read_csv(label_path, dtype={UID: str})
    required = [UID] + TARGETS
    required += [f"{target}__conf" for target in TARGETS]
    required += [f"{target}__verdict" for target in TARGETS]
    missing_columns = sorted(set(required) - set(labels.columns))
    if missing_columns:
        raise ValueError(f"label table is missing columns: {missing_columns}")
    if not labels[UID].is_unique or set(labels[UID]) != set(train[UID]):
        missing_ids = sorted(set(train[UID]) - set(labels[UID]))
        extra_ids = sorted(set(labels[UID]) - set(train[UID]))
        raise ValueError(
            f"label coverage mismatch: {len(missing_ids)} missing, {len(extra_ids)} extra; "
            f"missing={missing_ids[:10]}, extra={extra_ids[:10]}")

    train_by_uid = train.set_index(UID)
    labels_by_uid = labels.set_index(UID).loc[train_by_uid.index]
    blank_count = 0
    for target in TARGETS:
        raw = labels_by_uid[f"{target}__verdict"]
        missing = raw.isna()
        gold = pd.to_numeric(train_by_uid[target], errors="coerce")
        unbacked = missing & ~gold.notna()
        if unbacked.any():
            raise ValueError(
                f"{target}: blank verdicts without gold: "
                f"{labels_by_uid.index[unbacked].tolist()[:10]}")
        normalized = raw.fillna("UNK").astype(str).str.strip().str.upper()
        invalid = sorted(set(normalized.unique()) - {"YES", "NO", "UNK"})
        if invalid:
            raise ValueError(f"{target}: invalid verdicts: {invalid}")
        blank_count += int(missing.sum())

    label_sha = hashlib.sha256(label_path.read_bytes()).hexdigest()
    if len(labels) == 4407 and label_sha != EXPECTED_LABEL_SHA256:
        raise ValueError(
            f"label SHA-256 mismatch: {label_sha}; expected {EXPECTED_LABEL_SHA256}")
    print(f"preflight: mode={mode}, labels={len(labels)}, label_sha256={label_sha}, "
          f"gold-backed blanks->UNK={blank_count}")

print(f"RSNA_RUN2_PATH={os.environ['RSNA_RUN2_PATH']}")
print(f"RSNA_DINOV2_DIR={os.environ['RSNA_DINOV2_DIR']}")
if mode != "infer":
    print(f"RSNA_LABELS_PATH={os.environ['RSNA_LABELS_PATH']}")


In [ ]:
"""Prelude embedded into the generated run-3 hierarchical MIL Kaggle notebook."""
from __future__ import annotations

import gc
import hashlib
import io
import json
import math
import os
import re
import threading
import time
import traceback
import unicodedata
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_thread_var, "4")

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

cv2.setNumThreads(1)
T0 = time.time()


def log(message):
    print(f"[{time.time() - T0:7.1f}s] {message}", flush=True)


MODE = os.environ.get("RSNA_MODE", "").strip().lower()
if MODE not in {"train", "infer", "smoke"}:
    raise RuntimeError("RSNA_MODE must be exactly train, infer, or smoke")
ALLOW_LEXICON_DEBUG = os.environ.get("RSNA_ALLOW_LEXICON_DEBUG", "0") == "1"
if ALLOW_LEXICON_DEBUG and MODE != "smoke":
    raise RuntimeError("RSNA_ALLOW_LEXICON_DEBUG=1 is permitted only with RSNA_MODE=smoke")

UID = "StudyInstanceUID"
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]
N_TARGET = len(TARGETS)
TARGET_IDX = {target: index for index, target in enumerate(TARGETS)}
CONF_COLS = [target + "__conf" for target in TARGETS]
VERDICT_COLS = [target + "__verdict" for target in TARGETS]
LABEL_COLS = TARGETS + CONF_COLS + VERDICT_COLS
EXPECTED_LABEL_SHA256 = "13b0be74f9844c38181d096145fc39041e8e45c0b57243d83e2f5d01f29132c4"
EXPECTED_RUN2_ZIP_SHA256 = "96ed139413359eda2bc92a9aeb63ec7d4b07bc958a4f8ffc749073df43963fc2"

SEED = 2026
N_FOLDS = 5
EPOCHS = 10
MIN_EPOCHS = 4
EARLY_STOP_PATIENCE = 3
BATCH_STUDIES = 8
EVAL_BATCH = 8
LR_HEAD = 1e-3
LR_BACKBONE = 4e-6
UNFREEZE_LAST = 6
WEIGHT_DECAY = 0.02
GOLD_WEIGHT = 3.0
CONF_FLOOR = 0.25
CROP_MM = 130.0
GROUP = 3
IMG = 336
SLICE_BAND = (0.20, 0.80)
TIME_BUDGET = 8.0 * 3600
INFERENCE_RESERVE = 150 * 60
RETRY_MIN_REMAINING = 25 * 60
BOOTSTRAP_SAMPLES = 2000
RUN2_BLEND_WEIGHTS = (0.70, 0.30)
RUN3_BLEND_CANDIDATES = (0.15, 0.30, 0.45)
GOLD_DECLINE_LIMIT = 0.01
TRAIN_WINDOWS = 2
INFER_WINDOW_CHUNK = 2
TARGET_MASS_SCALE_LIMITS = (0.5, 3.0)
FINGERPRINT_TOL = 2e-3
# Kaggle exposes an exact three-study placeholder test set before the submitted notebook
# is rerun against the hidden evaluation set. Rank fusion can legitimately tie a target
# across all three placeholder rows; this narrow exemption never applies to a larger set.
VISIBLE_TEST_STUB_MAX_ROWS = 3
LAT_MIN_OFFSET_MM = 20.0
LAT_MIN_AGREEMENT = 0.70
LAT_WARN_AGREEMENT = 0.90
FG_THRESH = 8
FG_MARGIN = 4
HDR_THREADS = 16
ORDER_THREADS = 32
PIX_THREADS = 12
ORDER_BUDGET_S = 75 * 60
CACHE_MIN_FRAC = 0.05
POOL = "cls_mean_focal"
POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}
SLOT_PRIOR_STRENGTH = 0.55
SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
N_SLOT = len(SLOTS)
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
TTA_TARGET_POOL = {
    "Fracture": "max", "Contusion": "max", "Medial Meniscus": "max",
    "Lateral Meniscus": "max", "Baker's": "max", "ACL": "top2", "MCL": "top2",
}
RUN2_BANKS = [
    {"name": "detail", "img": 336, "cache_slices": 6, "group": 3,
     "slice_band": (0.20, 0.80), "seed": 2026, "expected_windows": 4},
    {"name": "depth", "img": 224, "cache_slices": 12, "group": 3,
     "slice_band": (0.12, 0.88), "seed": 3407, "expected_windows": 10},
]
HIERARCHICAL_BANK = {
    "name": "hierarchical", "img": 224, "cache_slices": 12, "group": 3,
    "slice_band": (0.12, 0.88), "seed": 7301, "expected_windows": 10,
}
BANKS = [HIERARCHICAL_BANK]

# Exact tokens: GE's SAT_GEMS is spatial saturation, not fat suppression.
FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


def stable_seed(*parts):
    payload = "\x1f".join(str(part) for part in parts).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big") % (2 ** 32 - 1)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def canonical_json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),
                                 ensure_ascii=True).encode()).hexdigest()


def set_global_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False


def find_root():
    explicit = os.environ.get("RSNA_INPUT_DIR", "").strip()
    candidates = ([Path(explicit)] if explicit else []) + [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"), Path("data"), Path("."),
    ]
    for candidate in candidates:
        if (candidate / "train.csv").is_file() and (candidate / "test.csv").is_file():
            return candidate
    base = Path("/kaggle/input")
    if base.is_dir():
        for first in sorted(path for path in base.iterdir() if path.is_dir()):
            for candidate in [first] + sorted(path for path in first.iterdir() if path.is_dir()):
                if (candidate / "train.csv").is_file() and (candidate / "test.csv").is_file():
                    return candidate
    raise FileNotFoundError("competition mount not found; set RSNA_INPUT_DIR explicitly")


def find_dinov2():
    explicit = os.environ.get("RSNA_DINOV2_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / "config.json").is_file():
            return path
        raise FileNotFoundError(f"RSNA_DINOV2_DIR has no config.json: {path}")
    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [name for name in dirs if name not in {"train_series", "test_series"}]
            if "config.json" in files and "dinov2" in root.lower() and "small" in root.lower():
                return Path(root)
    raise FileNotFoundError("offline DINOv2-S files not found; set RSNA_DINOV2_DIR")


ROOT = find_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
OUT_WEIGHTS = WORK / "weights"
OUT_WEIGHTS.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = OUT_WEIGHTS / "manifest.json"
RECEIPT_PATH = WORK / "run_receipt.json"
OOF_PATH = WORK / "oof_predictions.csv"
METRICS_PATH = WORK / "oof_metrics.json"
SUBMISSION_PATH = WORK / ("smoke_submission.csv" if MODE == "smoke" else "submission.csv")
ANCHOR_SUBMISSION_PATH = WORK / "submission_run2_anchor.csv"
RUN3_SUBMISSION_PATH = WORK / "submission_run3_only.csv"
CANDIDATE_SUBMISSION_PATHS = {
    weight: WORK / f"submission_anchor_run3_{int(round(weight * 100)):02d}.csv"
    for weight in RUN3_BLEND_CANDIDATES
}
TRAIN_DF = pd.read_csv(ROOT / "train.csv", dtype={UID: str})
TEST_DF = pd.read_csv(ROOT / "test.csv", dtype={UID: str})
for name, frame in (("train.csv", TRAIN_DF), ("test.csv", TEST_DF)):
    if UID not in frame or frame[UID].isna().any() or not frame[UID].is_unique:
        raise RuntimeError(f"{name} must contain unique, non-null {UID}")
if "Report" not in TRAIN_DF:
    raise RuntimeError("train.csv has no Report column")
if "Report" in TEST_DF:
    log("test Report column is present but is never read or used")

DINO_PATH = find_dinov2()
DEVS = ([torch.device(f"cuda:{index}") for index in range(torch.cuda.device_count())]
        if torch.cuda.is_available() else [torch.device("cpu")])
AMP = torch.cuda.is_available()
if MODE == "smoke":
    N_FOLDS, EPOCHS, MIN_EPOCHS, EARLY_STOP_PATIENCE = 2, 1, 1, 1
    BATCH_STUDIES, EVAL_BATCH = 2, 2
    TIME_BUDGET = 40 * 60
    INFERENCE_RESERVE = 8 * 60
    BOOTSTRAP_SAMPLES = 40
    ORDER_BUDGET_S = 5 * 60
    HDR_THREADS = ORDER_THREADS = PIX_THREADS = 2

log(f"contract: mode={MODE}, root={ROOT}, DINOv2={DINO_PATH}, devices={DEVS}, "
    f"folds={N_FOLDS}; run-2 anchor, labels, folds, and weights fail closed")


## Header pass and physical pixel contract

## §7 The pixel contract

Everything below this point exists to make one guarantee: **the same anatomy lands on the
same patch tokens regardless of which scanner, protocol or knee produced it.** Four
independent failures of that guarantee were each measurable in this repository, and each
has a rule here.

| what breaks it | what the corpus showed | the rule |
|---|---|---|
| **Scanner field of view** varies 70–320 mm, so a fixed-pixel resize makes 1 px mean anything from 0.2 to 1.4 mm | a 160 mm crop is *larger than the image* in 60% of series and is silently skipped for all of them | crop a fixed **130 mm** box, then resize — mm/px is constant at 0.39 |
| **Slice order on disk** is not anatomical order | filename order correlates with position along the slice normal at **ρ ≈ 0.01** | order by projecting `ImagePositionPatient` onto the slice normal |
| **Left knees are mirror images of right knees** | 4 of 12 targets name `Medial` or `Lateral`; a flip swaps them | canonicalise every study to a right knee **once**, and never flip in augmentation |
| **`ImagePositionPatient` is a corner, not the centre** | corner-x agrees with the `Laterality` tag 58.8% of the time; walking half the extent along the direction cosines to the image centre agrees **96.9%** | derive the side from the image centre, and only where the tag is missing |

Two further details are easy to get wrong and cost real AUC:

**Normalise per series, not per slice.** A knee stack has bright fluid on some slices and
none on others. Percentile-normalising each slice independently rescales those differences
away — it is the inter-slice contrast that says *effusion*. The window is taken over the
whole ordered volume.

**The foreground box recentres the crop; it does not resize it.** Roughly 15–20% of a knee
MRI is air, and it is tempting to crop to the tissue and resize. That restores per-study
scaling and throws away the constant mm/px the resolution argument rests on. So the
tissue bounding box only chooses *where* the fixed 130 mm box sits.

**A sanity check that is cheap and catches everything.** After canonicalisation the medial
compartment is on a fixed side of every sagittal stack. If a change to any rule above
breaks that, the four side-specific targets collapse toward 0.5 while the other eight hold
— which is exactly the signature to look for when the score moves and nothing else explains
it.

In [ ]:
# =============================================================================
# 9. The header pass: what each series is
#
# One header read per series, not per slice. Roughly 20k series against ~10^6 files, so
# this pass is minutes and the slice-ordering pass below is the expensive one.
# =============================================================================
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "ImageLaterality",
            "PixelSpacing", "Rows", "Columns", "RescaleSlope", "RescaleIntercept",
            # Position and orientation come from the same header probe() already opens, so
            # they cost nothing, and they are what recovers the side when the Laterality
            # tag is absent - which it is for a large share of studies here.
            "ImagePositionPatient", "ImageOrientationPatient"]


def _hdr_vec(s, n):
    """Parse a DICOM multi-value string as stored by probe(): floats joined by `|`."""
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def probe(item):
    """Read one series: its file list and the header of its middle slice.

    The middle slice, not the first. The first slice of a knee stack is often outside the
    joint, and on some protocols its geometry tags differ from the body of the series.
    """
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    """Every series directory of a split, with one header read per series.

    An absent split returns an empty frame *with the columns annotate expects*. Returning a
    bare DataFrame looks like the same thing and is not: the next call indexes
    `SeriesDescription` and raises KeyError, so the branch that exists to survive a missing
    split is what turns it into a crash.
    """
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    df = pd.DataFrame(rows)
    for c in HDR_TAGS:
        if c not in df.columns:
            df[c] = None
    return df


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header.

    These two axes decide which slot a series fills, and neither is a DICOM field. They are
    reconstructed from the description, from ScanOptions, and - where the description says
    nothing, which is common - from TR and TE directly.
    """
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # GE writes SAT_GEMS for spatial saturation, so ScanOptions must be matched as exact
    # tokens; a substring test on "SAT" fires on series that are not fat-suppressed.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1 = desc.str.contains(_T1_RX)
    t2 = desc.str.contains(_T2_RX)
    pdw = desc.str.contains(_PD_RX)

    # The description wins where it is explicit; TR/TE decide the rest. A short TR is T1, a
    # long TE is T2, and a long TR with a short TE is proton density. Note that these are
    # mutually exclusive branches evaluated in order - an earlier version tested a T1
    # signature (short TE and short TR) inside the fat-suppression rule as well, which made
    # the two predicates contradict each other and left the coronal T1 slot permanently
    # empty for every study.
    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


_PLANE_AXES = {0: "Sagittal", 1: "Coronal", 2: "Axial"}


def plane_from_iop(iop_str):
    """Plane from the direction cosines, for series the provided table does not name.

    The slice normal is the cross product of the two in-plane direction vectors; whichever
    patient axis it points along most strongly names the plane. This is a fallback: the
    competition ships an Anatomical_Plane column and it is used first.
    """
    v = _hdr_vec(iop_str, 6)
    if v is None:
        return None
    k = np.cross(v[:3], v[3:6])
    n = np.linalg.norm(k)
    if not np.isfinite(n) or n < 1e-6:
        return None
    return _PLANE_AXES[int(np.argmax(np.abs(k / n)))]


def build_plane_map(series_tables, headers):
    """SeriesInstanceUID -> plane, from the provided table with a geometry fallback."""
    pm = {}
    for t in series_tables:
        if t is not None and "Anatomical_Plane" in t.columns:
            pm.update(dict(zip(t["SeriesInstanceUID"], t["Anatomical_Plane"])))
    n_tab = sum(1 for v in pm.values() if isinstance(v, str))
    n_geo = 0
    for h in headers:
        if not len(h):
            continue
        for uid, iop in zip(h["SeriesInstanceUID"], h["ImageOrientationPatient"]):
            if not isinstance(pm.get(uid), str):
                p = plane_from_iop(iop)
                if p is not None:
                    pm[uid] = p
                    n_geo += 1
    log(f"plane: {n_tab} from the provided table, {n_geo} recovered from geometry")
    return pm


def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties break toward the stack with the most slices: a thicker stack samples the joint more
    densely, and the slice sampler benefits from the margin.

    A slot with no series matching its predicate stays EMPTY, and no substitute is admitted
    from a neighbouring predicate. Relaxing the weighting to fill a T1 slot draws from the
    pool SAG_FLUID_NOFS selects from, since that pool is what remains once the weighting is
    dropped: over the training corpus it puts one series in two slots for 2,383 of 4,407
    studies and leaves 56% of the T1 slot holding PD or T2. The presence mask would then
    assert a sequence that was never acquired, and the per-finding softmax would divide its
    attention across two identical slots, giving one acquisition about twice the weight it
    carries in a study that genuinely holds both. The mask exists to say a slot is absent,
    which is what an absent slot is.
    """
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs) & (g["fluid"] == fluid)
            cand = g[sel]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out

In [ ]:
# =============================================================================
# 10. Geometry: which knee, which order, which pixels
# =============================================================================


def side_from_geometry(h):
    """Study -> 'L' / 'R' / None, from where the image sits in the patient.

    `Laterality` (0020,0060) is Type 2C and may legitimately be absent; in this corpus it is
    missing on about half the studies, and the vendors it is missing from are whole vendors
    rather than scattered series. A study with no tag is not a left knee, but a
    normalisation that only acts on 'R' treats it as one - so half the corpus goes
    unnormalised and the five side-defined targets see that axis reversed on a large
    minority of it.

    The patient coordinate system fixes this without the tag. DICOM patient coordinates are
    LPS: +x is the patient's left, so the centre of a right knee sits at negative x. The
    CENTRE is used rather than `ImagePositionPatient` itself, because that tag is the corner
    of the image and is offset by half a field of view - enough to change the sign on a knee
    near the midline. Walking half the extent along the direction cosines is what moves
    agreement with the tag from 58.8% to 96.9%.

    The median over a study's series is thresholded, not a single series: probe() reads one
    arbitrary slice per series, which on a sagittal stack can sit anywhere across the joint.
    """
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = (ipp[:3]
                 + iop[:3] * ps[1] * float(cols) / 2
                 + iop[3:6] * ps[0] * float(rows) / 2)
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        # Studies whose centre falls near the midline are left unresolved rather than
        # guessed: inside 20 mm the rule is no better than chance.
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def lat_of(h, tag=""):
    """Study -> 'L' / 'R' / None: the tag where it exists, geometry where it does not.

    The tag comes first and always wins. Geometry is a fallback for the untagged half, and
    it has to earn that role: agreement with the tag is measured on the studies where both
    exist, and if it falls below the gate the fallback is refused outright rather than
    applied to studies that cannot check it.
    """
    geo = side_from_geometry(h)
    tagged, agree = {}, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        if "ImageLaterality" in g.columns:
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        # An empty string is not NaN and is not a side, so the first usable character is
        # what is read rather than the value being truthy.
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        if v:
            tagged[st] = v[0]
            if geo.get(st) is not None:
                agree += geo[st] == v[0]
    both = sum(1 for st in tagged if geo.get(st) is not None)
    rate = agree / both if both else 1.0
    trust = rate >= LAT_MIN_AGREEMENT or both < 50

    d, n_tag, n_geo, n_none = {}, 0, 0, 0
    for st in h["StudyInstanceUID"].unique():
        side = tagged.get(st)
        if side is not None:
            n_tag += 1
        else:
            side = geo.get(st) if trust else None
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} "
        f"unresolved; the two agree on {rate:.1%} of the {both} studies carrying both")
    if not trust:
        log(f"{tag}WARNING: geometry agreement {rate:.1%} is below the "
            f"{LAT_MIN_AGREEMENT:.0%} gate, so the fallback was REFUSED. The untagged "
            f"studies stay unnormalised, which costs the four side-specific targets.")
    elif rate < LAT_WARN_AGREEMENT:
        log(f"{tag}WARNING: geometry agreement {rate:.1%} is below the {LAT_WARN_AGREEMENT:.0%} "
            f"expected from this corpus; the fallback is still used but check the mount.")
    return d


# --------------------------------------------------------------------------- #
# slice order
# --------------------------------------------------------------------------- #
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

# Series in which at least one sampled slice would not decode. A list rather than a counter
# because appending is atomic under the reader threads, and reported rather than swallowed:
# unreported, a decode failure is indistinguishable from a black knee.
DECODE_FAILED = []
NO_GEOMETRY = []


def order_slices(rec):
    """Return the series' files sorted along the through-plane axis.

    A DICOM file name here is a SOP Instance UID, which is assigned arbitrarily. Sorting by
    it produces an order uncorrelated with anatomy - Spearman between file-name rank and
    physical position is about 0.01, i.e. none. Anything that assumes the file order means
    something is then operating on noise: the three channels of a 2.5D input are three
    unrelated views rather than neighbouring slices, "the middle of the stack" is a random
    subset, and reversing slice order to normalise laterality reverses nothing.

    The physical order is recoverable exactly. Each slice carries its position in patient
    coordinates and its two in-plane axes, so projecting the position onto the slice normal
    gives a signed through-plane coordinate that is monotonic along the stack:

        n = r_x  x  r_y ,      k = p . n

    `InstanceNumber` is the fallback. It usually tracks the projection up to sign, but
    interleaved and multi-echo acquisitions need not number slices in the order they occupy
    in space - and the projection is signed in patient coordinates, which is exactly what
    the laterality convention needs.
    """
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        ds = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None or not np.isfinite(k) for k, _ in keyed):
        # A series with no usable geometry keeps its arbitrary order. That is worse than
        # sorting and better than dropping the series, and it is counted so the run says how
        # often it happened instead of degrading silently.
        NO_GEOMETRY.append(rec.get("SeriesInstanceUID", d))
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


# --------------------------------------------------------------------------- #
# pixels
# --------------------------------------------------------------------------- #
def _fg_centre(vol, h, w):
    """Centre of the tissue bounding box, or the image centre when there is no tissue.

    Roughly 15-20% of a knee MRI is air, and the joint is not always centred in the field of
    view. This moves the fixed crop onto the tissue - it does NOT resize it. Cropping to the
    box and resizing would restore per-study scaling and throw away the constant mm/px that
    the whole resolution argument rests on.
    """
    m = vol.max(axis=0)
    thr = FG_THRESH / 255.0 * (m.max() if m.max() > 0 else 1.0)
    ys, xs = np.nonzero(m > thr)
    if len(ys) < 16:
        return h // 2, w // 2
    cy = int(np.clip((ys.min() + ys.max()) // 2, 0, h - 1))
    cx = int(np.clip((xs.min() + xs.max()) // 2, 0, w - 1))
    return cy, cx


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` physically spread slices from one series, at `out_size` pixels.

    Returns uint8 [n_slice, out, out], normalised per SERIES to its 1st-99th percentile.
    Percentiles rather than min/max because MR intensity has no absolute scale and one bright
    vessel would compress the whole dynamic range. Per series rather than per slice because
    a knee stack has bright fluid on some slices and none on others, and normalising each
    slice alone rescales exactly the inter-slice contrast that says `effusion`.

    Reading is the expensive half of this pipeline, so the caller reads once at the largest
    configuration it needs and derives the smaller ones from the returned buffer.
    """
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files = rec.get("ordered") or rec["files"]
    d, px = rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Spread the samples over a central band of the stack: the outermost slices of a knee
    # series are mostly soft tissue outside the joint. The band is a constant rather than a
    # literal because how much of the stack is worth reading depends on how many slices are
    # taken - at three the middle is all that fits, while at twelve the ends are worth
    # having, and a Baker cyst sits at the posteromedial end of a sagittal stack.
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = (np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo
           else np.array([n // 2]))
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
            if str(getattr(ds, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
                a = float(np.nanmax(a) + np.nanmin(a)) - a
        except Exception:
            a = None                      # no shape is known here; see below
        planes.append(a)

    # A slice that would not decode has no shape of its own, and inventing one is how a
    # single unreadable file erases a whole series: a substitute allocated at the resize
    # target while the decoded slices are still native makes the shape check below take the
    # substitute as the authority and zero the good slices with it, leaving a black slot that
    # the presence mask still reports as acquired.
    #
    # A failure is instead filled from the nearest slice that DID decode - the same
    # convention the sampler already uses when the band holds fewer distinct slices than were
    # asked for - and a series where nothing decodes is reported absent, which the mask can
    # express, rather than black, which it cannot.
    got = [k for k, p in enumerate(planes) if p is not None]
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    # Slices of one series can still differ in matrix size - multi-echo and some reformats
    # do - and those are genuinely not stackable.
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # Constant physical extent, then resize. PixelSpacing varies 3.4x across the corpus, so
    # without this step one pixel means something different in every study and a 14-pixel
    # patch token covers anywhere from 2.8 mm to 9.5 mm of knee.
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            half = want // 2
            cy, cx = _fg_centre(vol, h, w)
            # Keep the box inside the image and keep its SIZE fixed: clamping the centre
            # rather than truncating the box is what preserves the physical extent.
            cy = int(np.clip(cy, half, h - half))
            cx = int(np.clip(cx, half, w - half))
            vol = vol[:, cy - half:cy + half, cx - half:cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32. These buffers queue between the reader threads and the encoder, and
    # at this size a float32 slot-series is several megabytes. Intensity is already normalised
    # into [0, 1] here, so eight bits cost nothing a bilinear resize has not already cost,
    # and the cache is a quarter the size - which is what decides how many slices fit.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


def normalise_laterality(img, plane, lat):
    """Map every knee onto one canonical side, once, before anything learns from it.

    Coronal and axial views of a left and a right knee are mirror images in plane, so the
    horizontal axis is flipped - `dims=[-1]`, the LAST axis of a [slices, H, W] stack. (The
    vertical axis is not a mirror of anything: flipping it puts the tibia above the femur.)

    Sagittal stacks are not in-plane mirrors of each other at all. What differs is the
    direction the stack runs: medial-to-lateral one way for a left knee and the other way for
    a right one. So the SLICE order is reversed instead - `dims=[0]`.

    This is done once here so that no flip is ever needed in augmentation, and no flip may
    ever be added: four of the twelve targets are named `Medial ...` or `Lateral ...`, and a
    mirror swaps them.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])

In [ ]:
# =============================================================================
# 12. The model: one encoder, six slots, twelve queries
#
# DINOv2 ViT-S/14 is the encoder. It is a self-supervised model, which is what makes it work
# here: this task has 58 expert labels, so a supervised ImageNet head has nothing to
# fine-tune against, and a frozen ImageNet backbone scores 0.47 - indistinguishable from
# chance on most targets. The same backbone fine-tuned scores 0.866.
# =============================================================================
DINO_PATH = find_dinov2()


class SlotHead(nn.Module):
    """Twelve findings, each attending over the six acquisitions it can be seen in.

    A knee study is a BAG of acquisitions, not an image, and which acquisition answers a
    question depends on the question. An effusion is unmistakable on axial fluid-sensitive
    images and nearly invisible on coronal T1; a medial meniscal tear is the reverse. Mean- or
    max-pooling the bag applies one answer to all twelve, so every finding is read from a
    compromise view.

    So each finding gets its own query and its own softmax over the slots. Three details
    matter:

    * The softmax is MASKED by which slots the study actually has. About a third of studies
      are missing at least one acquisition, and a missing slot is a zero array - attending to
      it teaches the query what an empty buffer looks like.
    * A fixed anatomical prior tilts each query toward the slots that can physically show its
      finding (SLOT_PRIOR_TABLE). It is added to the logits, so at strength 0.55 a favoured
      slot is 1.73x as likely to be read and the head can still learn past it. It is part of
      the architecture and therefore has to be reproduced exactly for a stored state dict to
      mean anything.
    * A study with NO usable slot at all would make the masked softmax divide by zero. Those
      rows are unmasked over their zero features instead, which makes the output a constant
      per target - the learned prior, which is the right answer when there is no evidence.
    """

    def __init__(self, d_in, d_hidden=384, dropout=0.1):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, d_hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.zeros(N_SLOT, d_hidden))
        self.query = nn.Parameter(torch.randn(N_TARGET, d_hidden) * d_hidden ** -0.5)
        self.drop = nn.Dropout(dropout)
        # Per-target readout: one weight vector per finding over its own pooled feature, not
        # a shared Linear. The twelve findings do not share a decision boundary.
        self.out_w = nn.Parameter(torch.randn(N_TARGET, d_hidden) * d_hidden ** -0.5)
        self.out_b = nn.Parameter(torch.zeros(N_TARGET))

        prior = torch.zeros(N_TARGET, N_SLOT)
        for name, slots in SLOT_PRIOR_TABLE.items():
            for s in slots:
                prior[TARGET_IDX[name], s] = SLOT_PRIOR_STRENGTH
        # A buffer, not a parameter: it is fixed, and it must travel with the state dict so
        # that a checkpoint cannot be loaded against a different prior table.
        self.register_buffer("prior", prior)
        self.scale = d_hidden ** -0.5

    def forward(self, feat, mask):
        """feat [B, N_SLOT, d_in] -> logits [B, N_TARGET]; mask [B, N_SLOT] bool."""
        h = self.proj(feat) + self.slot_emb           # [B, S, H]
        h = self.drop(h)
        logits = torch.einsum("bsh,th->bts", h, self.query) * self.scale
        logits = logits + self.prior.unsqueeze(0)

        # Studies with nothing: attend uniformly over their zero features rather than over
        # nothing at all.
        m = mask
        empty = ~m.any(dim=1)
        if empty.any():
            m = m | empty.unsqueeze(1)
        logits = logits.masked_fill(~m.unsqueeze(1), float("-inf"))
        w = logits.softmax(dim=-1)                    # [B, T, S]

        pooled = torch.einsum("bts,bsh->bth", w, h)   # [B, T, H]
        return (pooled * self.out_w.unsqueeze(0)).sum(-1) + self.out_b


class Model(nn.Module):
    """The encoder, applied once per slot, and the head, applied once per study."""

    def __init__(self, backbone, d_model, pool=POOL, unfreeze_last=UNFREEZE_LAST):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(d_model * POOL_PARTS[pool])
        self._set_trainable(unfreeze_last)

    def _set_trainable(self, unfreeze_last):
        """Adapt the top of the encoder; leave the bottom alone.

        With 58 expert labels and ~4.3k noisy ones, fine-tuning 22M parameters end to end
        overfits the noise. The early blocks of a self-supervised ViT hold edge and texture
        filters that transfer as they are; only the last few need to learn that this is a
        knee. The final layernorm is unfrozen with them - it is what the pooled features are
        read through, and freezing it holds them at ImageNet-era statistics.
        """
        for p in self.backbone.parameters():
            p.requires_grad = False
        layers = getattr(getattr(self.backbone, "encoder", None), "layer", None)
        n_open = 0
        if layers is not None and unfreeze_last > 0:
            for blk in layers[-unfreeze_last:]:
                for p in blk.parameters():
                    p.requires_grad = True
                    n_open += p.numel()
        ln = getattr(self.backbone, "layernorm", None)
        if ln is not None:
            for p in ln.parameters():
                p.requires_grad = True
                n_open += p.numel()
        self.n_open = n_open

    def encode(self, x):
        """x [N, GROUP, H, W] -> [N, d_model * parts].

        The three slices of a group become the three input channels. They are neighbouring
        slices in physical order - which is the whole point of the ordering pass - so the
        encoder sees local through-plane context in the channel dimension for free, without
        the cost of a 3D backbone.
        """
        out = self.backbone(pixel_values=x)
        h = out.last_hidden_state                     # [N, 1 + P, D]
        cls, pat = h[:, 0], h[:, 1:]
        parts = [cls, pat.mean(dim=1)]
        if self.pool == "cls_mean_focal":
            # Most of a knee image is not the finding. A patch-mean over 576 tokens dilutes a
            # 1 mm tear across the whole joint; the mean of the most active eighth keeps a
            # focal signal legible while staying a smooth function of the input, which a hard
            # argmax is not. Selection is by token norm - parameter-free, so it is identical
            # across runs and safe to fingerprint.
            k = max(1, pat.shape[1] // 8)
            sel = pat.norm(dim=-1).topk(k, dim=1).indices
            parts.append(pat.gather(1, sel.unsqueeze(-1).expand(-1, -1, pat.shape[-1]))
                         .mean(dim=1))
        return torch.cat(parts, dim=-1)

    def forward(self, bag, mask):
        """bag [B, N_SLOT, GROUP, H, W] (float, ImageNet-normalised) -> [B, N_TARGET]."""
        b, s = bag.shape[:2]
        # Every slot of every study in one encoder call. Looping the slots would launch six
        # small kernels per study and leave the GPU idle between them.
        feat = self.encode(bag.reshape(b * s, *bag.shape[2:])).reshape(b, s, -1)
        return self.head(feat, mask)


def build_model(device):
    """Load the encoder from the attached weights directory and wrap it in the head.

    `AutoModel` rather than `Dinov2Model`, and no `add_pooling_layer` argument: the exact
    transformers version on the image is not under this notebook's control, internet is off,
    and a keyword that a neighbouring release renamed would fail here - after the cache is
    built. The pooling layer this leaves behind is never called; `encode` reads
    `last_hidden_state` directly.
    """
    from transformers import AutoModel
    backbone = AutoModel.from_pretrained(DINO_PATH)
    d = backbone.config.hidden_size
    m = Model(backbone, d).to(device)
    log(f"model: DINOv2 d={d}, pool={POOL} ({POOL_PARTS[POOL]}x{d}={d * POOL_PARTS[POOL]}), "
        f"{m.n_open / 1e6:.1f}M encoder params open of "
        f"{sum(p.numel() for p in backbone.parameters()) / 1e6:.1f}M, "
        f"head {sum(p.numel() for p in m.head.parameters()) / 1e6:.2f}M")
    return m


# ImageNet statistics, because that is what DINOv2 was trained under. The images are
# single-channel MR replicated across a 3-slice group, so all three channels get the same
# statistics - which is correct: they are the same modality, not R, G and B.
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

## Supervision, hierarchical training, OOF gate, and staged inference

`YES` and `NO` report observations retain their soft targets; `UNK` has exactly zero loss.
Gold labels override the loss target and remain a separate diagnostic. The exact run-2 fold
mapping is reproduced and the matching depth backbone warm-starts each new fold.

Epoch selection uses explicit-label OOF. Up to three strong late checkpoints are averaged
and retained only when the average improves explicit-label OOF without violating the gold
safety limit. Test caches are staged as run-2 detail then shared depth; the verified anchor
submission is written before the new bank can vote.


In [ ]:
"""Run-3 hierarchical MIL training, anchored inference, and evidence receipt code."""

DECODE_FAILED = []
_MODEL_INIT_LOCK = threading.Lock()


class LabelSourceError(RuntimeError):
    pass


class CacheError(RuntimeError):
    pass


class PackageError(RuntimeError):
    pass


class Run2Bundle:
    """Read the verified run-2 package from either a directory or the original zip."""

    def __init__(self, path):
        self.path = Path(path)
        self._zip = None
        if self.path.is_file() and self.path.suffix.lower() == ".zip":
            self._zip = zipfile.ZipFile(self.path)
            corrupt = self._zip.testzip()
            if corrupt is not None:
                raise PackageError(f"run-2 zip CRC failure: {corrupt}")
            self.bundle_sha256 = sha256_file(self.path)
        elif self.path.is_dir():
            self.bundle_sha256 = None
        else:
            raise PackageError(f"RSNA_RUN2_PATH must be a directory or zip file: {self.path}")

    def _directory_path(self, relative):
        relative = Path(relative)
        candidates = [self.path / relative]
        if relative.parts and relative.parts[0] == "weights":
            candidates.append(self.path / Path(*relative.parts[1:]))
        for candidate in candidates:
            if candidate.is_file():
                return candidate
        raise PackageError(f"run-2 bundle member is missing: {relative}")

    def read_bytes(self, relative):
        relative = str(relative).replace("\\", "/")
        if self._zip is not None:
            try:
                return self._zip.read(relative)
            except KeyError as error:
                raise PackageError(f"run-2 bundle member is missing: {relative}") from error
        return self._directory_path(relative).read_bytes()

    def sha256(self, relative):
        relative = str(relative).replace("\\", "/")
        digest = hashlib.sha256()
        if self._zip is not None:
            try:
                handle = self._zip.open(relative)
            except KeyError as error:
                raise PackageError(f"run-2 bundle member is missing: {relative}") from error
            with handle:
                for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                    digest.update(block)
        else:
            path = self._directory_path(relative)
            with path.open("rb") as handle:
                for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                    digest.update(block)
        return digest.hexdigest()

    def json(self, relative):
        return json.loads(self.read_bytes(relative).decode("utf-8"))

    def dataframe(self, relative, **kwargs):
        return pd.read_csv(io.BytesIO(self.read_bytes(relative)), **kwargs)

    def torch_state(self, relative):
        return torch.load(io.BytesIO(self.read_bytes(relative)), map_location="cpu")


def resolve_run2_bundle():
    explicit = os.environ.get("RSNA_RUN2_PATH", "").strip()
    if not explicit:
        raise PackageError("RSNA_RUN2_PATH is required and must point to results.zip or its extracted directory")
    return Run2Bundle(explicit)


def verify_run2_bundle(bundle):
    if MODE != "smoke" and bundle.bundle_sha256 is not None and \
            bundle.bundle_sha256 != EXPECTED_RUN2_ZIP_SHA256:
        raise PackageError(
            f"run-2 zip SHA-256 mismatch: {bundle.bundle_sha256}; "
            f"expected {EXPECTED_RUN2_ZIP_SHA256}")
    manifest = bundle.json("weights/manifest.json")
    receipt = bundle.json("run_receipt.json")
    if manifest.get("schema_version") != 2 or manifest.get("recipe") != "training-first-two-bank-v1":
        raise PackageError("run-2 manifest schema/recipe mismatch")
    if manifest.get("selected_recipe") != "detail_depth_70_30":
        raise PackageError("run-2 anchor is not the verified 70/30 recipe")
    if manifest.get("targets") != TARGETS or manifest.get("slots") != [slot[0] for slot in SLOTS]:
        raise PackageError("run-2 target/slot order mismatch")
    bundle_folds = int(manifest.get("n_folds", -1))
    if bundle_folds != N_FOLDS and not (MODE == "smoke" and bundle_folds == 5):
        raise PackageError(f"run-2 fold count {bundle_folds} != active {N_FOLDS}")
    if receipt.get("label_sha256") != manifest.get("label_sha256"):
        raise PackageError("run-2 receipt and manifest label hashes disagree")
    if bundle.sha256("weights/manifest.json") != receipt.get("manifest_sha256"):
        raise PackageError("run-2 manifest SHA-256 mismatch")
    if bundle.sha256("report_labels_v2_complete.csv") != manifest.get("label_sha256"):
        raise PackageError("run-2 bundled label SHA-256 mismatch")
    for bank in RUN2_BANKS:
        stored = manifest.get("banks", {}).get(bank["name"], {})
        members = sorted(stored.get("members", []), key=lambda item: int(item["fold"]))
        if not stored.get("complete") or [int(item["fold"]) for item in members] != list(range(bundle_folds)):
            raise PackageError(f"run-2 {bank['name']} bank is incomplete")
        for member in members:
            relative = f"weights/{member['file']}"
            if bundle.sha256(relative) != member.get("weight_sha256"):
                raise PackageError(f"run-2 checkpoint SHA-256 mismatch: {relative}")
    log(f"run-2 bundle verified: selected=detail_depth_70_30, "
        f"label={manifest['label_sha256'][:16]}..., bundle={bundle.bundle_sha256 or 'directory'}")
    return manifest, receipt


def normalise_report(value):
    text = unicodedata.normalize("NFKD", "" if pd.isna(value) else str(value))
    text = "".join(char for char in text if not unicodedata.combining(char)).lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text)).strip()


def report_group_keys(frame):
    keys = []
    for uid, report in zip(frame[UID].astype(str), frame["Report"]):
        clean = normalise_report(report)
        token = clean if clean else f"__empty__:{uid}"
        keys.append(hashlib.sha256(token.encode()).hexdigest())
    return np.asarray(keys, dtype=object)


def _debug_labels(frame):
    if MODE != "smoke" or not ALLOW_LEXICON_DEBUG:
        raise LabelSourceError("debug labels require RSNA_MODE=smoke and RSNA_ALLOW_LEXICON_DEBUG=1")
    rows = []
    for uid, report in zip(frame[UID].astype(str), frame["Report"].fillna("")):
        clean = normalise_report(report)
        row = {UID: uid}
        for target in TARGETS:
            # Deterministic synthetic shakeout labels only. They are deliberately not a
            # production lexicon and smoke mode cannot write submission.csv.
            score = 0.15 + 0.70 * (stable_seed(uid, clean, target, "debug-label") / (2 ** 32 - 1))
            row[target] = score
            row[target + "__conf"] = 0.25
            row[target + "__verdict"] = "YES" if score > 0.5 else "NO"
        rows.append(row)
    return pd.DataFrame(rows), "debug-smoke-only"


def load_supervision(frame):
    """Load a complete public label table and keep teacher, loss, and gold arrays separate."""
    explicit = os.environ.get("RSNA_LABELS_PATH", "").strip()
    if not explicit:
        table, label_sha = _debug_labels(frame)
        label_path = None
    else:
        label_path = Path(explicit)
        if not label_path.is_file():
            raise LabelSourceError(f"RSNA_LABELS_PATH is not a file: {label_path}")
        label_sha = sha256_file(label_path)
        table = pd.read_csv(label_path, dtype={UID: str})

    required = [UID] + LABEL_COLS
    missing = [column for column in required if column not in table]
    if missing:
        raise LabelSourceError(f"label table is missing required columns: {missing}")
    if table[UID].isna().any() or not table[UID].is_unique:
        raise LabelSourceError(f"label table {UID} must be unique and non-null")
    train_ids = frame[UID].astype(str)
    table_ids = table[UID].astype(str)
    missing_ids = sorted(set(train_ids) - set(table_ids))
    extra_ids = sorted(set(table_ids) - set(train_ids))
    if missing_ids or extra_ids:
        missing_preview = missing_ids[:10]
        extra_preview = extra_ids[:10]
        raise LabelSourceError(
            f"label coverage must exactly match train.csv: {len(missing_ids)} missing, "
            f"{len(extra_ids)} extra; missing IDs={missing_preview}; extra IDs={extra_preview}. "
            "Do not invent a production label: regenerate the table with exact coverage. "
            "For a pipeline-only smoke test, unset RSNA_LABELS_PATH and set "
            "RSNA_ALLOW_LEXICON_DEBUG=1.")
    table = table.set_index(UID).loc[train_ids]
    numeric_columns = TARGETS + CONF_COLS
    values = table[numeric_columns].to_numpy(dtype=np.float64)
    if not np.isfinite(values).all():
        raise LabelSourceError("label table contains NaN or infinite target/confidence values")
    if (values < 0).any() or (values > 1).any():
        raise LabelSourceError("every target/confidence value must lie in [0,1]")

    # The verified run-2 table contains one coverage-repair study whose supervision comes
    # entirely from official gold. Its report verdicts are blank by construction. Normalize
    # only gold-backed blanks to UNK; every other missing or malformed verdict remains fatal.
    if all(target in frame for target in TARGETS):
        candidate_gold = frame[TARGETS].apply(
            pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    else:
        candidate_gold = np.full((len(frame), N_TARGET), np.nan, dtype=np.float32)
    raw_verdict = table[VERDICT_COLS]
    missing_verdict = raw_verdict.isna().to_numpy()
    unbacked_missing = missing_verdict & ~np.isfinite(candidate_gold)
    if unbacked_missing.any():
        locations = np.argwhere(unbacked_missing)[:10]
        preview = [(str(train_ids.iloc[row]), TARGETS[column]) for row, column in locations]
        raise LabelSourceError(
            f"missing verdicts are allowed only for official-gold overrides: {preview}")
    verdict = raw_verdict.fillna("UNK").astype(str).apply(
        lambda column: column.str.strip().str.upper())
    invalid_verdicts = sorted(set(verdict.to_numpy().ravel()) - {"YES", "NO", "UNK"})
    if invalid_verdicts:
        raise LabelSourceError(f"label verdicts must be YES, NO, or UNK: {invalid_verdicts}")
    if MODE != "smoke" and len(frame) == 4407 and label_sha != EXPECTED_LABEL_SHA256:
        raise LabelSourceError(
            f"production label SHA-256 mismatch: {label_sha}; expected {EXPECTED_LABEL_SHA256}")

    teacher_y = table[TARGETS].to_numpy(dtype=np.float32)
    teacher_conf = table[CONF_COLS].to_numpy(dtype=np.float32)
    known_mask = verdict.to_numpy() != "UNK"
    # Preserve the exact run-2 receipt calculation for lineage verification. Its original
    # `!= UNK` expression counted the twelve NaNs in the gold-only repair row as known.
    # Run-3 selection uses known_mask above, which is strictly explicit YES/NO supervision.
    legacy_known_mask = known_mask | missing_verdict
    loss_y = teacher_y.copy()
    loss_conf = teacher_conf.copy()
    gold_y = np.full_like(teacher_y, np.nan)
    gold_mask = np.zeros_like(teacher_y, dtype=bool)
    if all(target in frame for target in TARGETS):
        finite = np.isfinite(candidate_gold)
        if finite.any() and ((candidate_gold[finite] < 0).any() or
                             (candidate_gold[finite] > 1).any()):
            raise LabelSourceError("official labels in train.csv must lie in [0,1]")
        gold_y[finite] = candidate_gold[finite]
        gold_mask[finite] = True
        loss_y[finite] = candidate_gold[finite]
        loss_conf[finite] = 1.0
    loss_mask = known_mask | gold_mask
    log(f"labels: {label_path or 'debug smoke generator'}; SHA-256={label_sha}; "
        f"exact coverage={len(table)}, gold values={int(gold_mask.sum())}, "
        f"gold-backed blank verdicts normalized to UNK={int(missing_verdict.sum())}")
    return {
        "teacher_y": teacher_y, "teacher_conf": teacher_conf, "known_mask": known_mask,
        "legacy_known_mask": legacy_known_mask,
        "loss_y": loss_y, "loss_conf": loss_conf,
        "loss_mask": loss_mask, "gold_y": gold_y, "gold_mask": gold_mask,
        "verdict": verdict.to_numpy(), "label_sha256": label_sha,
    }


def load_run2_oof(bundle, frame, supervision):
    required = [UID, "report_group_sha256", "fold"]
    for target in TARGETS:
        required.extend([f"detail__{target}", f"depth__{target}"])
    oof = bundle.dataframe("oof_predictions.csv", dtype={UID: str})
    missing = [column for column in required if column not in oof]
    if missing:
        raise PackageError(f"run-2 OOF table is missing columns: {missing}")
    if oof[UID].isna().any() or not oof[UID].is_unique:
        raise PackageError("run-2 OOF IDs must be unique and non-null")
    expected_ids = list(frame[UID].astype(str))
    if set(oof[UID]) != set(expected_ids):
        raise PackageError("run-2 OOF coverage does not exactly match active training studies")
    oof = oof.set_index(UID).loc[expected_ids].reset_index()
    groups = report_group_keys(frame)
    if not np.array_equal(oof["report_group_sha256"].astype(str).to_numpy(), groups.astype(str)):
        raise PackageError("run-2 normalized report-group hashes do not match")
    folds = pd.to_numeric(oof["fold"], errors="coerce").to_numpy()
    if not np.isfinite(folds).all() or not np.equal(folds, folds.astype(int)).all():
        raise PackageError("run-2 fold column is not integral")
    folds = folds.astype(np.int16)
    run2_fold_count = len(set(folds))
    if sorted(set(folds)) != list(range(run2_fold_count)):
        raise PackageError("run-2 folds are not contiguous from zero")
    if (pd.DataFrame({"group": groups, "fold": folds}).groupby("group").fold.nunique() > 1).any():
        raise PackageError("run-2 fold mapping leaks a normalized report group")
    detail = oof[[f"detail__{target}" for target in TARGETS]].to_numpy(np.float32)
    depth = oof[[f"depth__{target}" for target in TARGETS]].to_numpy(np.float32)
    if not np.isfinite(detail).all() or not np.isfinite(depth).all():
        raise PackageError("run-2 OOF predictions are incomplete")
    anchor = RUN2_BLEND_WEIGHTS[0] * percentile_rank(detail) + RUN2_BLEND_WEIGHTS[1] * percentile_rank(depth)
    known = macro_from_metrics(per_target_auc(
        supervision["teacher_y"], anchor, supervision["known_mask"]))
    legacy_known = macro_from_metrics(per_target_auc(
        supervision["teacher_y"], anchor, supervision["legacy_known_mask"]))
    all_teacher = macro_from_metrics(per_target_auc(supervision["teacher_y"], anchor))
    gold = macro_from_metrics(per_target_auc(
        supervision["gold_y"], anchor, supervision["gold_mask"], min_pos=1))
    log(f"run-2 OOF anchor reproduced: strict-YES/NO={known:.5f}, "
        f"legacy-receipt={legacy_known:.5f}, all={all_teacher:.5f}, gold={gold:.5f}")
    if MODE != "smoke" and len(frame) == 4407:
        expected = (0.865116, 0.864792, 0.839587, 0.840538)
        observed = (known, legacy_known, all_teacher, gold)
        if max(abs(a - b) for a, b in zip(observed, expected)) > 2e-5:
            raise PackageError(f"run-2 OOF metric reproduction mismatch: {observed}")
    return {"frame": oof, "folds": folds, "groups": groups,
            "detail": detail, "depth": depth, "anchor": anchor,
            "known_macro_auc": known, "legacy_known_macro_auc": legacy_known,
            "all_teacher_macro_auc": all_teacher,
            "gold_macro_auc": gold}


def assign_grouped_multilabel_folds(frame, teacher_y, gold_mask, n_folds=N_FOLDS, seed=SEED):
    """Deterministic report-grouped greedy multilabel stratification.

    The objective balances study count, positive and negative soft-target mass, and
    per-target gold mass. All iteration and tie breaking is SHA-256 based, so changing
    PYTHONHASHSEED cannot alter a fold.
    """
    groups = report_group_keys(frame)
    members = {}
    for index, key in enumerate(groups):
        members.setdefault(str(key), []).append(index)
    feature_rows = {}
    for key, indices in members.items():
        idx = np.asarray(indices, dtype=int)
        pos = teacher_y[idx].sum(axis=0, dtype=np.float64)
        neg = (1.0 - teacher_y[idx]).sum(axis=0, dtype=np.float64)
        gold = gold_mask[idx].sum(axis=0, dtype=np.float64)
        feature_rows[key] = np.r_[len(idx), pos, neg, gold]
    total = np.stack(list(feature_rows.values())).sum(axis=0)
    target = total / n_folds
    scale = np.maximum(target, 1.0)

    def priority(key):
        vector = feature_rows[key]
        rarity = float(np.max(vector[1:] / np.maximum(total[1:], 1.0)))
        return (-rarity, -vector[0], hashlib.sha256(f"{seed}:{key}".encode()).hexdigest())

    fold_mass = np.zeros((n_folds, len(target)), dtype=np.float64)
    assignment = {}
    for key in sorted(feature_rows, key=priority):
        vector = feature_rows[key]
        scores = []
        for fold in range(n_folds):
            trial = fold_mass.copy()
            trial[fold] += vector
            imbalance = float(np.square((trial - target) / scale).sum())
            tie = hashlib.sha256(f"{seed}:{key}:{fold}".encode()).hexdigest()
            scores.append((imbalance, fold_mass[fold, 0], tie, fold))
        chosen = min(scores)[-1]
        assignment[key] = chosen
        fold_mass[chosen] += vector
    folds = np.asarray([assignment[str(key)] for key in groups], dtype=np.int16)
    if any(len(set(folds[np.asarray(indices)])) != 1 for indices in members.values()):
        raise AssertionError("identical normalized reports crossed folds")
    counts = [int((folds == fold).sum()) for fold in range(n_folds)]
    if any(count == 0 for count in counts):
        raise RuntimeError(f"fold stratifier produced an empty fold: {counts}")
    log(f"folds: {len(members)} report groups, study counts={counts}; no report leakage")
    return folds, groups


def bank_config(bank):
    return {key: bank[key] for key in
            ("name", "img", "cache_slices", "group", "slice_band", "seed", "expected_windows")}


def bank_fingerprint(bank):
    payload = {
        "bank": bank_config(bank), "crop_mm": CROP_MM, "slots": [slot[0] for slot in SLOTS],
        "targets": TARGETS, "architecture": "target-patch-slot-window-mil-v1",
        "pool": POOL, "unfreeze_last": UNFREEZE_LAST,
        "lr_head": LR_HEAD, "lr_backbone": LR_BACKBONE,
        "slot_prior_strength": SLOT_PRIOR_STRENGTH, "train_windows": TRAIN_WINDOWS,
        "infer_window_chunk": INFER_WINDOW_CHUNK, "unknown_loss_weight": 0.0,
        "target_mass_scale_limits": list(TARGET_MASS_SCALE_LIMITS),
        "warm_start": "matching-run2-depth-backbone-only", "no_horizontal_flip": True,
    }
    return canonical_json_hash(payload)


def activate_bank(bank):
    global IMG, GROUP, SLICE_BAND
    IMG = int(bank["img"])
    GROUP = int(bank["group"])
    SLICE_BAND = tuple(float(value) for value in bank["slice_band"])
    starts = window_starts(int(bank["cache_slices"]), GROUP)
    if len(starts) != int(bank["expected_windows"]):
        raise RuntimeError(f"{bank['name']} window contract broken: {starts}")
    return starts


def smoke_train_uids(frame, supervision):
    if MODE != "smoke":
        return list(frame[UID].astype(str))
    gold_studies = supervision["gold_mask"].any(axis=1)
    gold_ids = list(frame.loc[gold_studies, UID].astype(str))
    remaining = list(frame.loc[~gold_studies, UID].astype(str))
    rng = np.random.RandomState(stable_seed("smoke-studies"))
    take = min(max(24 - len(gold_ids), 8), len(remaining))
    sampled = [remaining[index] for index in rng.permutation(len(remaining))[:take]]
    chosen = list(dict.fromkeys(gold_ids + sampled))
    log(f"smoke subset: {len(chosen)} train studies including {len(gold_ids)} with gold labels")
    return chosen


def order_all(slots_by_study, tag=""):
    """Geometry-order every selected series; malformed series become absent, never fallback."""
    records, seen = [], set()
    for chosen in slots_by_study.values():
        for row in chosen.values():
            series_uid = row["SeriesInstanceUID"]
            if series_uid not in seen:
                seen.add(series_uid)
                records.append(row)
    started = time.time()
    output, stats = {}, {"expired": 0, "malformed": 0}
    lock = threading.Lock()

    def one(row):
        if time.time() - started > ORDER_BUDGET_S:
            with lock:
                stats["expired"] += 1
            return
        files, ordered = order_slices({"files": row["files"], "dir": row["dir"],
                                      "SeriesInstanceUID": row["SeriesInstanceUID"]})
        with lock:
            if ordered:
                output[row["SeriesInstanceUID"]] = files
            else:
                stats["malformed"] += 1

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        list(pool.map(one, records))
    log(f"{tag}slice order: {len(output)} geometry-ordered, "
        f"{stats['malformed']} malformed and masked absent, {stats['expired']} expired")
    return output, stats


def prepare_shared_metadata(train_uids=None):
    """Read headers and geometry/order metadata once; pixel caches remain bank-staged."""
    include_train = train_uids is not None
    h_train = annotate(walk("train_series")) if include_train else None
    h_test = annotate(walk("test_series"))
    if include_train:
        keep = set(train_uids)
        h_train = h_train[h_train[UID].astype(str).isin(keep)].reset_index(drop=True)

    def read_optional(name):
        path = ROOT / name
        return pd.read_csv(path) if path.is_file() else None

    frames = ([read_optional("train_series.csv"), read_optional("test_series.csv")]
              if include_train else [read_optional("test_series.csv")])
    headers = ([h_train, h_test] if include_train else [h_test])
    plane_map = build_plane_map(frames, headers)
    result = {}
    for split, header, uids in (("train", h_train, train_uids),
                                ("test", h_test, list(TEST_DF[UID].astype(str)))):
        if header is None:
            continue
        lat = lat_of(header, f"{split} ")
        slots = pick_slots(header, plane_map)
        order, order_stats = order_all(slots, f"{split} ")
        if order_stats["expired"]:
            raise CacheError(f"{split}: geometry ordering deadline expired for "
                             f"{order_stats['expired']} selected series")
        result[split] = {"uids": list(uids), "slots": slots, "lat": lat, "order": order}
    return result


def available_ram_bytes():
    try:
        return int(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_AVPHYS_PAGES"))
    except (AttributeError, OSError, ValueError):
        return 8 * 1024 ** 3


def allocate_exact_cache(shape, tag):
    """Allocate exactly `shape`; use a disk memmap rather than reducing depth."""
    nbytes = int(np.prod(shape, dtype=np.int64))
    force = os.environ.get("RSNA_FORCE_MEMMAP", "0") == "1"
    use_memmap = force or nbytes > int(0.38 * available_ram_bytes())
    path = None
    if not use_memmap:
        try:
            array = np.zeros(shape, dtype=np.uint8)
            return array, {"storage": "ram", "path": None, "bytes": nbytes}
        except (MemoryError, OSError):
            use_memmap = True
    if use_memmap:
        cache_dir = WORK / "pixel_cache"
        cache_dir.mkdir(parents=True, exist_ok=True)
        path = cache_dir / f"{tag}-{canonical_json_hash(list(shape))[:12]}.uint8.memmap"
        array = np.memmap(path, dtype=np.uint8, mode="w+", shape=shape)
        array[:] = 0
        log(f"{tag}: exact cache uses disk memmap {path}")
        return array, {"storage": "memmap", "path": str(path), "bytes": nbytes}
    raise AssertionError("unreachable")


def build_exact_cache(meta, bank, split, deadline=None):
    activate_bank(bank)
    studies = list(meta["uids"])
    n_slice, out_size = int(bank["cache_slices"]), int(bank["img"])
    shape = (len(studies), N_SLOT, n_slice, out_size, out_size)
    cache, allocation = allocate_exact_cache(shape, f"{bank['name']}-{split}")
    mask = np.zeros((len(studies), N_SLOT), dtype=bool)
    index = {uid: row for row, uid in enumerate(studies)}
    lock = threading.Lock()
    state = {"expired": False, "slots": 0}

    def decode_study(uid):
        if deadline is not None and time.time() >= deadline:
            with lock:
                state["expired"] = True
            return
        row_index = index[uid]
        chosen = meta["slots"].get(uid, {})
        side = meta["lat"].get(uid)
        for slot_index, (name, plane, _fluid, _fat_sat) in enumerate(SLOTS):
            row = chosen.get(name)
            if row is None:
                continue
            ordered = meta["order"].get(row["SeriesInstanceUID"])
            if not ordered:
                continue
            record = {"files": row["files"], "dir": row["dir"], "px": row["px"],
                      "SeriesInstanceUID": row["SeriesInstanceUID"],
                      "ordered": ordered}
            image = read_slot(record, n_slice=n_slice, out_size=out_size)
            if image is None:
                continue
            image = normalise_laterality(image, plane, side)
            cache[row_index, slot_index] = image.numpy()
            mask[row_index, slot_index] = True
            with lock:
                state["slots"] += 1

    log(f"{bank['name']} {split}: decode exact shape={shape}, "
        f"{allocation['bytes'] / 1024 ** 3:.2f} GiB")
    try:
        with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
            list(pool.map(decode_study, studies))
    except Exception:
        release_cache(cache, allocation)
        raise
    if isinstance(cache, np.memmap):
        cache.flush()
    if state["expired"]:
        release_cache(cache, allocation)
        raise CacheError(f"{bank['name']} {split}: decode deadline expired; partial cache discarded")
    have = mask.any(axis=1)
    fraction = float(have.mean()) if len(have) else 0.0
    if fraction < CACHE_MIN_FRAC:
        release_cache(cache, allocation)
        raise CacheError(f"{bank['name']} {split}: only {fraction:.1%} of studies decoded a slot")
    log(f"{bank['name']} {split}: {int(mask.sum())} slots; "
        f"{int(have.sum())}/{len(have)} studies represented; storage={allocation['storage']}")
    return {"pixels": cache, "mask": mask, "allocation": allocation,
            "uids": studies, "bank": bank_config(bank), "split": split}


def release_cache(cache, allocation):
    path = allocation.get("path") if allocation else None
    if isinstance(cache, np.memmap):
        cache.flush()
        mmap = getattr(cache, "_mmap", None)
        if mmap is not None:
            mmap.close()
    del cache
    gc.collect()
    if path:
        try:
            Path(path).unlink()
        except FileNotFoundError:
            pass


def window_starts(n_slice, group=GROUP):
    if n_slice < group:
        raise ValueError("cache must contain at least one complete slice group")
    return list(range(n_slice - group + 1))


def augment_rigid(x, rng):
    """Small rigid/intensity jitter. There is intentionally no image-flip operation."""
    count = x.shape[0]
    angle = rng.uniform(-8.0, 8.0, count) * np.pi / 180.0
    scale = rng.uniform(0.92, 1.08, count)
    tx = rng.uniform(-0.05, 0.05, count)
    ty = rng.uniform(-0.05, 0.05, count)
    theta = np.zeros((count, 2, 3), dtype=np.float32)
    theta[:, 0, 0] = scale * np.cos(angle)
    theta[:, 0, 1] = -scale * np.sin(angle)
    theta[:, 1, 0] = scale * np.sin(angle)
    theta[:, 1, 1] = scale * np.cos(angle)
    theta[:, 0, 2], theta[:, 1, 2] = tx, ty
    theta_t = torch.from_numpy(theta).to(x.device)
    grid = F.affine_grid(theta_t, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    gain = torch.from_numpy(rng.uniform(0.90, 1.10, (count, 1, 1, 1)).astype(np.float32)).to(x.device)
    return (x * gain).clamp_(0, 1)


IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def to_input(chunk, device, rng=None):
    x = torch.from_numpy(np.ascontiguousarray(chunk)).to(device, non_blocking=True).float().div_(255)
    batch, slots, group = x.shape[:3]
    x = x.reshape(batch * slots, group, *x.shape[3:])
    if rng is not None:
        x = augment_rigid(x, rng)
    x = (x - IMNET_MEAN.to(device)) / IMNET_STD.to(device)
    return x.reshape(batch, slots, group, *x.shape[2:])


# The archived 0.877 notebook defines these immediately before this runtime cell. Keep an
# explicit alias so run-2 anchor loading cannot accidentally instantiate the new head.
Run2Model = Model
Run2SlotHead = SlotHead
_build_run2_model = build_model


def window_centres(starts, n_slice, group=GROUP):
    starts = np.asarray(starts, dtype=np.float32)
    denominator = max(float(n_slice - 1), 1.0)
    return (starts + (group - 1) / 2.0) / denominator


def paired_window_starts(batch_size, starts, rng):
    """Two independently sampled windows conditioned on wide circular separation."""
    starts = np.asarray(starts, dtype=np.int64)
    if len(starts) < 2:
        raise ValueError("hierarchical training needs at least two windows")
    first_index = rng.randint(0, len(starts), size=int(batch_size))
    minimum = max(1, len(starts) // 2 - 1)
    second_index = np.empty_like(first_index)
    all_indices = np.arange(len(starts))
    for row, first in enumerate(first_index):
        separation = np.abs(all_indices - first)
        circular = np.minimum(separation, len(starts) - separation)
        eligible = all_indices[circular >= minimum]
        second_index[row] = eligible[rng.randint(0, len(eligible))]
    return np.stack([starts[first_index], starts[second_index]], axis=1)


def gather_windows(cache, rows, starts_by_row):
    rows = np.asarray(rows, dtype=np.int64)
    starts_by_row = np.asarray(starts_by_row, dtype=np.int64)
    if starts_by_row.shape[0] != len(rows):
        raise ValueError("one window-start row is required per study")
    bags = []
    for row, row_starts in zip(rows, starts_by_row):
        bags.append(np.stack([cache[row, :, start:start + GROUP] for start in row_starts], axis=0))
    return np.stack(bags, axis=0)


def to_window_input(chunk, device, rng=None):
    """uint8 [B,W,S,3,H,W] -> normalized float tensor with the same leading axes."""
    x = torch.from_numpy(np.ascontiguousarray(chunk)).to(device, non_blocking=True).float().div_(255)
    batch, windows, slots, group = x.shape[:4]
    x = x.reshape(batch * windows * slots, group, *x.shape[4:])
    if rng is not None:
        x = augment_rigid(x, rng)
    x = (x - IMNET_MEAN.to(device)) / IMNET_STD.to(device)
    return x.reshape(batch, windows, slots, group, *x.shape[2:])


class HierarchicalMILHead(nn.Module):
    """Target-aware patch, acquisition-slot, and ordered-window aggregation."""

    def __init__(self, d_model, d_hidden=384, dropout=0.10):
        super().__init__()
        self.d_model = int(d_model)
        self.d_hidden = int(d_hidden)
        self.base_proj = nn.Sequential(
            nn.LayerNorm(3 * d_model), nn.Linear(3 * d_model, d_hidden), nn.GELU())
        self.patch_norm = nn.LayerNorm(d_model)
        self.patch_query = nn.Parameter(torch.randn(N_TARGET, d_model) * d_model ** -0.5)
        self.patch_proj = nn.Sequential(nn.Linear(d_model, d_hidden), nn.GELU())
        self.target_emb = nn.Parameter(torch.randn(N_TARGET, d_hidden) * d_hidden ** -0.5)
        self.slot_emb = nn.Parameter(torch.zeros(N_SLOT, d_hidden))
        self.slot_query = nn.Parameter(torch.randn(N_TARGET, d_hidden) * d_hidden ** -0.5)
        self.position = nn.Sequential(
            nn.Linear(1, d_hidden), nn.GELU(), nn.Linear(d_hidden, d_hidden))
        self.window_query = nn.Parameter(torch.randn(N_TARGET, d_hidden) * d_hidden ** -0.5)
        self.drop = nn.Dropout(dropout)
        self.out_norm = nn.LayerNorm(d_hidden)
        self.out_w = nn.Parameter(torch.randn(N_TARGET, d_hidden) * d_hidden ** -0.5)
        self.out_b = nn.Parameter(torch.zeros(N_TARGET))
        prior = torch.zeros(N_TARGET, N_SLOT)
        for name, slot_indices in SLOT_PRIOR_TABLE.items():
            for slot_index in slot_indices:
                prior[TARGET_IDX[name], slot_index] = SLOT_PRIOR_STRENGTH
        self.register_buffer("prior", prior)
        self.patch_scale = d_model ** -0.5
        self.hidden_scale = d_hidden ** -0.5

    def patch_features(self, cls, patches):
        k = max(1, patches.shape[1] // 8)
        indices = patches.norm(dim=-1).topk(k, dim=1).indices
        focal = patches.gather(
            1, indices.unsqueeze(-1).expand(-1, -1, patches.shape[-1])).mean(dim=1)
        common = self.base_proj(torch.cat([cls, patches.mean(dim=1), focal], dim=-1))
        normalized = self.patch_norm(patches)
        patch_score = torch.einsum("npd,td->ntp", normalized, self.patch_query) * self.patch_scale
        patch_attention = patch_score.softmax(dim=-1)
        target_patch = torch.einsum("ntp,npd->ntd", patch_attention, patches)
        target_patch = self.patch_proj(target_patch)
        return common.unsqueeze(1) + target_patch + self.target_emb.unsqueeze(0)

    def aggregate_slots(self, target_features, present):
        # target_features [B,W,S,T,H], present [B,S]
        h = target_features + self.slot_emb.view(1, 1, N_SLOT, 1, self.d_hidden)
        h = self.drop(h)
        scores = torch.einsum("bwsth,th->bwts", h, self.slot_query) * self.hidden_scale
        scores = scores + self.prior.view(1, 1, N_TARGET, N_SLOT)
        mask = present
        empty = ~mask.any(dim=1)
        if empty.any():
            mask = mask | empty.unsqueeze(1)
        scores = scores.masked_fill(~mask[:, None, None, :], float("-inf"))
        attention = scores.softmax(dim=-1)
        return torch.einsum("bwts,bwsth->bwth", attention, h)

    def aggregate_windows(self, window_features, positions, window_mask=None):
        # window_features [B,W,T,H], positions [B,W]
        h = window_features + self.position(positions.unsqueeze(-1)).unsqueeze(2)
        scores = torch.einsum("bwth,th->btw", self.drop(h), self.window_query) * self.hidden_scale
        if window_mask is not None:
            scores = scores.masked_fill(~window_mask[:, None, :], float("-inf"))
        attention = scores.softmax(dim=-1)
        pooled = torch.einsum("btw,bwth->bth", attention, h)
        pooled = self.out_norm(pooled)
        return (pooled * self.out_w.unsqueeze(0)).sum(dim=-1) + self.out_b


class HierarchicalModel(nn.Module):
    def __init__(self, backbone, d_model, unfreeze_last=UNFREEZE_LAST):
        super().__init__()
        self.backbone = backbone
        self.head = HierarchicalMILHead(d_model)
        self._set_trainable(unfreeze_last)

    def _set_trainable(self, unfreeze_last):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False
        layers = getattr(getattr(self.backbone, "encoder", None), "layer", None)
        n_open = 0
        if layers is not None and unfreeze_last > 0:
            for block in layers[-unfreeze_last:]:
                for parameter in block.parameters():
                    parameter.requires_grad = True
                    n_open += parameter.numel()
        layernorm = getattr(self.backbone, "layernorm", None)
        if layernorm is not None:
            for parameter in layernorm.parameters():
                parameter.requires_grad = True
                n_open += parameter.numel()
        self.n_open = n_open

    def encode_window_features(self, bag, present):
        # bag [B,W,S,3,H,W]
        batch, windows, slots = bag.shape[:3]
        encoded = self.backbone(pixel_values=bag.reshape(batch * windows * slots, *bag.shape[3:]))
        tokens = encoded.last_hidden_state
        target = self.head.patch_features(tokens[:, 0], tokens[:, 1:])
        target = target.reshape(batch, windows, slots, N_TARGET, self.head.d_hidden)
        return self.head.aggregate_slots(target, present)

    def aggregate_windows(self, window_features, positions, window_mask=None):
        return self.head.aggregate_windows(window_features, positions, window_mask)

    def forward(self, bag, present, positions, window_mask=None):
        features = self.encode_window_features(bag, present)
        return self.aggregate_windows(features, positions, window_mask)


def build_hierarchical_model(device):
    from transformers import AutoModel
    backbone = AutoModel.from_pretrained(DINO_PATH)
    model = HierarchicalModel(backbone, backbone.config.hidden_size).to(device)
    log(f"model: hierarchical DINOv2-S d={backbone.config.hidden_size}, "
        f"{model.n_open / 1e6:.1f}M encoder params open, "
        f"new head={sum(p.numel() for p in model.head.parameters()) / 1e6:.2f}M")
    return model


def build_seeded_model(device, seed):
    with _MODEL_INIT_LOCK:
        torch.manual_seed(seed)
        if device.type == "cuda":
            torch.cuda.manual_seed_all(seed)
        return build_hierarchical_model(device)


def warm_start_backbone(model, bundle, run2_manifest, fold):
    members = run2_manifest["banks"]["depth"]["members"]
    member = next((item for item in members if int(item["fold"]) == int(fold)), None)
    if member is None:
        raise PackageError(f"run-2 depth fold {fold} is missing")
    relative = f"weights/{member['file']}"
    if bundle.sha256(relative) != member.get("weight_sha256"):
        raise PackageError(f"run-2 warm-start checkpoint hash mismatch: {relative}")
    state = bundle.torch_state(relative)
    backbone_state = {key[len("backbone."):]: value for key, value in state.items()
                      if key.startswith("backbone.")}
    rejected = [key for key in state if not key.startswith("backbone.")]
    if not backbone_state or not rejected:
        raise PackageError("warm start must contain both a backbone and an excluded old head")
    model.backbone.load_state_dict(backbone_state, strict=True)
    loaded_names = {f"backbone.{key}" for key in backbone_state}
    if any(name.startswith("head.") for name in loaded_names):
        raise AssertionError("a run-2 head tensor entered the run-3 warm start")
    return {"source_member": member["name"], "source_weight_sha256": member["weight_sha256"],
            "loaded_backbone_tensors": len(backbone_state), "excluded_head_tensors": len(rejected)}


def target_window_pool(per_window):
    result = np.empty(per_window.shape[1:], dtype=np.float32)
    for target_index, target in enumerate(TARGETS):
        values = per_window[:, :, target_index]
        method = TTA_TARGET_POOL.get(target, "mean")
        if method == "max":
            result[:, target_index] = values.max(axis=0)
        elif method == "top2":
            result[:, target_index] = np.sort(values, axis=0)[-min(2, len(values)):].mean(axis=0)
        else:
            result[:, target_index] = values.mean(axis=0)
    return result


@torch.inference_mode()
def predict_member(model, cache_bundle, device, rows=None, tag=""):
    cache, mask = cache_bundle["pixels"], cache_bundle["mask"]
    n = len(cache) if rows is None else len(rows)
    starts = window_starts(cache.shape[2], GROUP)
    per_window = np.zeros((len(starts), n, N_TARGET), dtype=np.float32)
    model.eval()
    for window_index, start in enumerate(starts):
        for begin in range(0, n, EVAL_BATCH):
            selection = slice(begin, min(begin + EVAL_BATCH, n)) if rows is None \
                else rows[begin:begin + EVAL_BATCH]
            bag = to_input(cache[selection][:, :, start:start + GROUP], device)
            present = torch.from_numpy(mask[selection]).to(device)
            with torch.autocast("cuda", enabled=AMP):
                logits = model(bag, present)
            per_window[window_index, begin:begin + len(logits)] = logits.float().sigmoid().cpu().numpy()
    log(f"{tag}predicted {n} studies with {len(starts)} matched overlapping windows")
    return target_window_pool(per_window)


@torch.inference_mode()
def predict_hierarchical_member(model, cache_bundle, device, rows=None, tag=""):
    cache, present_mask = cache_bundle["pixels"], cache_bundle["mask"]
    selected_rows = np.arange(len(cache), dtype=int) if rows is None else np.asarray(rows, dtype=int)
    starts = window_starts(cache.shape[2], GROUP)
    centres = window_centres(starts, cache.shape[2], GROUP).astype(np.float32)
    output = np.zeros((len(selected_rows), N_TARGET), dtype=np.float32)
    model.eval()
    for begin in range(0, len(selected_rows), EVAL_BATCH):
        batch_rows = selected_rows[begin:begin + EVAL_BATCH]
        present = torch.from_numpy(present_mask[batch_rows]).to(device)
        feature_chunks = []
        for chunk_begin in range(0, len(starts), INFER_WINDOW_CHUNK):
            chunk_starts = starts[chunk_begin:chunk_begin + INFER_WINDOW_CHUNK]
            starts_by_row = np.tile(np.asarray(chunk_starts, dtype=int), (len(batch_rows), 1))
            bag = to_window_input(gather_windows(cache, batch_rows, starts_by_row), device)
            with torch.autocast("cuda", enabled=AMP):
                feature_chunks.append(model.encode_window_features(bag, present))
        features = torch.cat(feature_chunks, dim=1)
        positions = torch.from_numpy(np.tile(centres, (len(batch_rows), 1))).to(device)
        with torch.autocast("cuda", enabled=AMP):
            logits = model.aggregate_windows(features, positions)
        output[begin:begin + len(batch_rows)] = logits.float().sigmoid().cpu().numpy()
    log(f"{tag}predicted {len(selected_rows)} studies with learned attention over {len(starts)} windows")
    return output


def auc_one(y, prediction, mask=None, min_pos=3):
    y = np.asarray(y)
    prediction = np.asarray(prediction)
    valid = np.isfinite(y) & np.isfinite(prediction)
    if mask is not None:
        valid &= np.asarray(mask, dtype=bool)
    y, prediction = y[valid], prediction[valid]
    positive, negative = y > 0.5, y <= 0.5
    n_pos, n_neg = int(positive.sum()), int(negative.sum())
    if n_pos < min_pos or n_neg < min_pos:
        return float("nan")
    ranks = pd.Series(prediction).rank(method="average").to_numpy()
    return float((ranks[positive].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def per_target_auc(y, prediction, mask=None, min_pos=3):
    metrics = {}
    for index, target in enumerate(TARGETS):
        col_mask = None if mask is None else mask[:, index]
        metrics[target] = auc_one(y[:, index], prediction[:, index], col_mask, min_pos=min_pos)
    return metrics


def macro_from_metrics(metrics):
    values = [value for value in metrics.values() if np.isfinite(value)]
    return float(np.mean(values)) if values else float("nan")


def sample_weights(confidence, gold_mask):
    weights = CONF_FLOOR + (1.0 - CONF_FLOOR) * np.clip(confidence, 0, 1)
    return (weights * np.where(gold_mask, GOLD_WEIGHT, 1.0)).astype(np.float32)


def target_mass_scales(weights, loss_mask, rows):
    effective = (weights[rows] * loss_mask[rows]).sum(axis=0, dtype=np.float64)
    positive = effective[effective > 0]
    if len(positive) != N_TARGET:
        missing = [TARGETS[index] for index, value in enumerate(effective) if value <= 0]
        raise LabelSourceError(f"training fold has no supervised mass for targets: {missing}")
    reference = float(np.median(positive))
    lower, upper = TARGET_MASS_SCALE_LIMITS
    return np.clip(reference / effective, lower, upper).astype(np.float32)


def run2_model_fingerprint(model, device, bank):
    generator = torch.Generator().manual_seed(1234)
    bag = torch.rand((1, N_SLOT, GROUP, int(bank["img"]), int(bank["img"])), generator=generator)
    mask = torch.ones((1, N_SLOT), dtype=torch.bool)
    was_training = model.training
    model.eval()
    with torch.inference_mode():
        output = model(bag.to(device), mask.to(device)).float().cpu().numpy().ravel()
    if was_training:
        model.train()
    return [round(float(value), 6) for value in output]


def hierarchical_model_fingerprint(model, device, bank):
    generator = torch.Generator().manual_seed(1234)
    bag = torch.rand((1, TRAIN_WINDOWS, N_SLOT, GROUP, int(bank["img"]), int(bank["img"])),
                     generator=generator)
    present = torch.ones((1, N_SLOT), dtype=torch.bool)
    starts = window_starts(int(bank["cache_slices"]), GROUP)[:TRAIN_WINDOWS]
    positions = torch.tensor(window_centres(starts, int(bank["cache_slices"]), GROUP)).view(1, -1)
    was_training = model.training
    model.eval()
    with torch.inference_mode():
        output = model(bag.to(device), present.to(device), positions.to(device)).float().cpu().numpy().ravel()
    if was_training:
        model.train()
    return [round(float(value), 6) for value in output]


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=AMP)


def clone_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def average_states(states):
    if not states:
        raise ValueError("cannot average an empty state list")
    result = {}
    for key in states[0]:
        values = [state[key] for state in states]
        if values[0].is_floating_point():
            result[key] = torch.stack([value.float() for value in values]).mean(dim=0).to(values[0].dtype)
        else:
            result[key] = values[0].clone()
    return result


def checkpoint_average_accepted(best_metrics, averaged_metrics):
    """Known-label OOF must improve; gold can only veto a harmful average."""
    best_known = float(best_metrics["known_teacher_macro_auc"])
    averaged_known = float(averaged_metrics["known_teacher_macro_auc"])
    best_gold = float(best_metrics["gold_macro_auc"])
    averaged_gold = float(averaged_metrics["gold_macro_auc"])
    known_better = np.isfinite(averaged_known) and averaged_known > best_known + 1e-7
    gold_safe = (not np.isfinite(best_gold) or not np.isfinite(averaged_gold) or
                 averaged_gold >= best_gold - GOLD_DECLINE_LIMIT)
    return bool(known_better and gold_safe), bool(gold_safe)


def fold_metrics(supervision, rows, prediction):
    known = per_target_auc(supervision["teacher_y"][rows], prediction,
                           supervision["known_mask"][rows])
    all_teacher = per_target_auc(supervision["teacher_y"][rows], prediction)
    gold = per_target_auc(supervision["gold_y"][rows], prediction,
                          supervision["gold_mask"][rows], min_pos=1)
    return {"known_teacher_per_target_auc": known,
            "known_teacher_macro_auc": macro_from_metrics(known),
            "all_teacher_per_target_auc": all_teacher,
            "all_teacher_macro_auc": macro_from_metrics(all_teacher),
            "gold_per_target_auc": gold, "gold_macro_auc": macro_from_metrics(gold)}


def fit_fold(bank, fold, cache_bundle, supervision, folds, device, deadline,
             run2_bundle, run2_manifest):
    activate_bank(bank)
    train_rows = np.where(folds != fold)[0]
    valid_rows = np.where(folds == fold)[0]
    if len(train_rows) < max(4, BATCH_STUDIES) or len(valid_rows) < 2:
        raise RuntimeError(f"fold {fold} is too small: {len(train_rows)}/{len(valid_rows)}")
    seed = stable_seed(bank["seed"], fold, "hierarchical-training")
    rng = np.random.RandomState(seed)
    model = build_seeded_model(device, seed)
    warm_start = warm_start_backbone(model, run2_bundle, run2_manifest, fold)
    head_params = [parameter for parameter in model.head.parameters() if parameter.requires_grad]
    backbone_params = [parameter for parameter in model.backbone.parameters() if parameter.requires_grad]
    optimiser = torch.optim.AdamW(
        [{"params": head_params, "lr": LR_HEAD},
         {"params": backbone_params, "lr": LR_BACKBONE}], weight_decay=WEIGHT_DECAY)
    steps_per_epoch = max(1, math.ceil(len(train_rows) / BATCH_STUDIES))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimiser, max_lr=[LR_HEAD, LR_BACKBONE],
        total_steps=steps_per_epoch * EPOCHS, pct_start=0.15)
    scaler = make_scaler()
    base_weights = sample_weights(supervision["loss_conf"], supervision["gold_mask"])
    scales = target_mass_scales(base_weights, supervision["loss_mask"], train_rows)
    effective_weights = base_weights * scales.reshape(1, -1)
    starts = window_starts(cache_bundle["pixels"].shape[2], GROUP)
    best_auc, best_epoch = -np.inf, -1
    best_state, best_prediction, best_metrics = None, None, None
    late_candidates, history = [], []
    for epoch in range(EPOCHS):
        if time.time() >= deadline:
            log(f"{bank['name']} f{fold}: deadline before epoch {epoch}")
            break
        model.train()
        order = rng.permutation(len(train_rows))
        losses = []
        for begin in range(0, len(train_rows), BATCH_STUDIES):
            rows = train_rows[order[begin:begin + BATCH_STUDIES]]
            starts_by_row = paired_window_starts(len(rows), starts, rng)
            bag = to_window_input(
                gather_windows(cache_bundle["pixels"], rows, starts_by_row), device, rng)
            positions = torch.from_numpy(window_centres(
                starts_by_row, cache_bundle["pixels"].shape[2], GROUP).astype(np.float32)).to(device)
            present = torch.from_numpy(cache_bundle["mask"][rows]).to(device)
            target = torch.from_numpy(supervision["loss_y"][rows]).to(device)
            mask = torch.from_numpy(supervision["loss_mask"][rows]).to(device)
            weights = torch.from_numpy(effective_weights[rows]).to(device) * mask
            with torch.autocast("cuda", enabled=AMP):
                logits = model(bag, present, positions)
                element = F.binary_cross_entropy_with_logits(logits.float(), target, reduction="none")
                loss = (element * weights).sum() / weights.sum().clamp(min=1e-6)
            optimiser.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimiser)
            torch.nn.utils.clip_grad_norm_(
                [parameter for group in optimiser.param_groups for parameter in group["params"]], 1.0)
            scaler.step(optimiser)
            scaler.update()
            scheduler.step()
            losses.append(float(loss.detach()))
        prediction = predict_hierarchical_member(
            model, cache_bundle, device, rows=valid_rows, tag=f"{bank['name']} f{fold} e{epoch} ")
        metrics = fold_metrics(supervision, valid_rows, prediction)
        known_macro = metrics["known_teacher_macro_auc"]
        record = {"epoch": epoch, "loss": float(np.mean(losses)), **metrics}
        history.append(record)
        log(f"{bank['name']} f{fold} epoch {epoch}: loss={np.mean(losses):.5f}, "
            f"known={known_macro:.5f}, all={metrics['all_teacher_macro_auc']:.5f}, "
            f"gold-diagnostic={metrics['gold_macro_auc']:.5f}")
        state = None
        known_improved = np.isfinite(known_macro) and known_macro > best_auc + 1e-7
        gold_veto = False
        if known_improved and best_metrics is not None:
            previous_gold = best_metrics["gold_macro_auc"]
            candidate_gold = metrics["gold_macro_auc"]
            gold_veto = bool(np.isfinite(previous_gold) and np.isfinite(candidate_gold) and
                             candidate_gold < previous_gold - GOLD_DECLINE_LIMIT)
        record["gold_vetoed_known_improvement"] = gold_veto
        if known_improved and not gold_veto:
            state = clone_state(model)
            best_auc, best_epoch = known_macro, epoch
            best_state, best_prediction, best_metrics = state, prediction.copy(), metrics
        if epoch + 1 >= MIN_EPOCHS and np.isfinite(known_macro):
            state = state or clone_state(model)
            late_candidates.append({"epoch": epoch, "score": known_macro, "state": state})
            late_candidates = sorted(
                late_candidates, key=lambda item: (-item["score"], item["epoch"]))[:3]
        if epoch + 1 >= MIN_EPOCHS and epoch - best_epoch >= EARLY_STOP_PATIENCE:
            log(f"{bank['name']} f{fold}: early stop after three non-improving epochs")
            break
    if len(history) < MIN_EPOCHS:
        raise RuntimeError(
            f"{bank['name']} f{fold}: only {len(history)} epochs completed before the deadline; "
            f"at least {MIN_EPOCHS} are required")
    if best_state is None:
        raise RuntimeError(f"{bank['name']} f{fold}: no finite explicit-label teacher checkpoint")

    selected_state = best_state
    selected_prediction = best_prediction
    selected_metrics = best_metrics
    selection = {"type": "single", "epochs": [int(best_epoch)], "soup_evaluated": False}
    if len(late_candidates) >= 2:
        soup_state = average_states([item["state"] for item in late_candidates])
        model.load_state_dict(soup_state)
        soup_prediction = predict_hierarchical_member(
            model, cache_bundle, device, rows=valid_rows, tag=f"{bank['name']} f{fold} soup ")
        soup_metrics = fold_metrics(supervision, valid_rows, soup_prediction)
        soup_gold = soup_metrics["gold_macro_auc"]
        soup_accepted, gold_safe = checkpoint_average_accepted(best_metrics, soup_metrics)
        selection.update({
            "soup_evaluated": True,
            "soup_epochs": [int(item["epoch"]) for item in late_candidates],
            "soup_known_teacher_macro_auc": soup_metrics["known_teacher_macro_auc"],
            "soup_gold_macro_auc": soup_gold, "soup_gold_safe": bool(gold_safe),
        })
        if soup_accepted:
            selected_state, selected_prediction, selected_metrics = soup_state, soup_prediction, soup_metrics
            selection["type"] = "soup"
            selection["epochs"] = selection["soup_epochs"]
    model.load_state_dict(selected_state)
    return (model, valid_rows, selected_prediction, selected_metrics, best_epoch,
            history, selection, scales, warm_start)


def save_fold(model, device, bank, fold, metrics, best_epoch, history,
              selection, scales, warm_start):
    name = f"{bank['name']}_fold{fold}"
    path = OUT_WEIGHTS / f"{name}.pt"
    torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, path)
    entry = {
        "name": name, "bank": bank["name"], "fold": int(fold), "file": path.name,
        "weight_sha256": sha256_file(path), "config_fingerprint": bank_fingerprint(bank),
        "model_fingerprint": hierarchical_model_fingerprint(model, device, bank),
        "known_teacher_macro_auc": float(metrics["known_teacher_macro_auc"]),
        "all_teacher_macro_auc": float(metrics["all_teacher_macro_auc"]),
        "gold_macro_auc": float(metrics["gold_macro_auc"]),
        "best_single_epoch": int(best_epoch), "checkpoint_selection": selection,
        "target_mass_scales": {target: float(scales[index]) for index, target in enumerate(TARGETS)},
        "warm_start": warm_start,
        "history": history, "img": int(bank["img"]), "cache_slices": int(bank["cache_slices"]),
        "group": int(bank["group"]), "slice_band": list(bank["slice_band"]),
        "pool": POOL, "unfreeze_last": UNFREEZE_LAST, "targets": TARGETS,
        "slots": [slot[0] for slot in SLOTS],
    }
    log(f"saved {name}: {path.name}, SHA-256={entry['weight_sha256'][:16]}...")
    return entry


def _run_fold_queue(bank, wanted_folds, cache_bundle, supervision, folds, deadline,
                    run2_bundle, run2_manifest):
    results = {}
    queue = list(wanted_folds)
    lock = threading.Lock()

    def worker(device):
        while True:
            with lock:
                if not queue:
                    return
                fold = queue.pop(0)
            try:
                (model, rows, prediction, metrics, best_epoch, history,
                 selection, scales, warm_start) = fit_fold(
                    bank, fold, cache_bundle, supervision, folds, device, deadline,
                    run2_bundle, run2_manifest)
                member = save_fold(model, device, bank, fold, metrics, best_epoch, history,
                                   selection, scales, warm_start)
                result = {"ok": True, "fold": fold, "rows": rows, "prediction": prediction,
                          "member": member}
                del model
            except Exception:
                result = {"ok": False, "fold": fold, "error": traceback.format_exc()}
                log(f"{bank['name']} fold {fold} FAILED:\n{result['error']}")
            finally:
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()
            with lock:
                results[fold] = result

    threads = [threading.Thread(target=worker, args=(device,), daemon=True) for device in DEVS]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return results


def discard_bank_members(results):
    for result in results.values():
        if result.get("ok"):
            try:
                (OUT_WEIGHTS / result["member"]["file"]).unlink()
            except FileNotFoundError:
                pass


def train_bank(bank, meta, supervision, folds, deadline, run2_bundle, run2_manifest):
    activate_bank(bank)
    cache_bundle = build_exact_cache(meta, bank, "train", deadline=deadline)
    try:
        results = _run_fold_queue(bank, list(range(N_FOLDS)), cache_bundle,
                                  supervision, folds, deadline, run2_bundle, run2_manifest)
        failed = [fold for fold in range(N_FOLDS) if not results.get(fold, {}).get("ok")]
        if failed and time.time() + RETRY_MIN_REMAINING < deadline:
            log(f"{bank['name']}: retrying failed folds once: {failed}")
            retry = _run_fold_queue(bank, failed, cache_bundle, supervision, folds, deadline,
                                    run2_bundle, run2_manifest)
            results.update(retry)
        complete = all(results.get(fold, {}).get("ok") for fold in range(N_FOLDS))
        if not complete:
            discard_bank_members(results)
            return {"name": bank["name"], "complete": False, "members": [],
                    "completed_folds": sorted(fold for fold, result in results.items() if result.get("ok")),
                    "errors": {str(fold): result.get("error", "missing") for fold, result in results.items()
                               if not result.get("ok")}}
        oof = np.full((len(folds), N_TARGET), np.nan, dtype=np.float32)
        members = []
        for fold in range(N_FOLDS):
            result = results[fold]
            oof[result["rows"]] = result["prediction"]
            members.append(result["member"])
        if not np.isfinite(oof).all():
            discard_bank_members(results)
            raise RuntimeError(f"{bank['name']}: OOF matrix is incomplete")
        return {"name": bank["name"], "complete": True, "members": members,
                "completed_folds": list(range(N_FOLDS)), "oof": oof, "errors": {}}
    finally:
        release_cache(cache_bundle["pixels"], cache_bundle["allocation"])


def percentile_rank(prediction):
    prediction = np.asarray(prediction)
    ranked = np.empty_like(prediction, dtype=np.float64)
    for target in range(prediction.shape[1]):
        ranked[:, target] = pd.Series(prediction[:, target]).rank(method="average").to_numpy()
    return ranked / (len(prediction) + 1.0)


def equal_fold_rank_mean(predictions):
    if not predictions:
        raise RuntimeError("no fold predictions to combine")
    return np.mean([percentile_rank(prediction) for prediction in predictions], axis=0)


def evaluate_run3_blend_gate(anchor, run3, supervision, groups, folds,
                             candidates=RUN3_BLEND_CANDIDATES):
    anchor_rank, run3_rank = percentile_rank(anchor), percentile_rank(run3)
    known_mask = supervision["known_mask"]
    anchor_known = macro_from_metrics(per_target_auc(
        supervision["teacher_y"], anchor_rank, known_mask))
    anchor_gold = macro_from_metrics(per_target_auc(
        supervision["gold_y"], anchor_rank, supervision["gold_mask"], min_pos=1))
    unique_groups = sorted(set(groups))
    indices = {key: np.where(groups == key)[0] for key in unique_groups}
    evaluations = []
    for weight in candidates:
        blend = (1.0 - weight) * anchor_rank + weight * run3_rank
        candidate_known = macro_from_metrics(per_target_auc(
            supervision["teacher_y"], blend, known_mask))
        candidate_gold = macro_from_metrics(per_target_auc(
            supervision["gold_y"], blend, supervision["gold_mask"], min_pos=1))
        rng = np.random.RandomState(stable_seed(
            "run3-report-group-bootstrap", BOOTSTRAP_SAMPLES, weight))
        deltas = []
        for _ in range(BOOTSTRAP_SAMPLES):
            sampled = rng.choice(unique_groups, size=len(unique_groups), replace=True)
            rows = np.concatenate([indices[key] for key in sampled])
            base = macro_from_metrics(per_target_auc(
                supervision["teacher_y"][rows], anchor_rank[rows], known_mask[rows]))
            trial = macro_from_metrics(per_target_auc(
                supervision["teacher_y"][rows], blend[rows], known_mask[rows]))
            if np.isfinite(base) and np.isfinite(trial):
                deltas.append(trial - base)
        lower = float(np.percentile(deltas, 2.5)) if deltas else float("nan")
        fold_wins = 0
        fold_metrics = []
        for fold in range(N_FOLDS):
            rows = folds == fold
            base = macro_from_metrics(per_target_auc(
                supervision["teacher_y"][rows], anchor_rank[rows], known_mask[rows]))
            trial = macro_from_metrics(per_target_auc(
                supervision["teacher_y"][rows], blend[rows], known_mask[rows]))
            fold_metrics.append({"fold": fold, "anchor": base, "candidate": trial,
                                 "delta": trial - base})
            fold_wins += int(np.isfinite(base) and np.isfinite(trial) and trial > base)
        known_delta = candidate_known - anchor_known
        gold_available = bool(np.isfinite(anchor_gold) and np.isfinite(candidate_gold))
        gold_delta = candidate_gold - anchor_gold if gold_available else float("nan")
        promote = bool(known_delta > 0 and np.isfinite(lower) and lower >= 0 and
                       fold_wins >= max(1, N_FOLDS - 1) and gold_available and
                       gold_delta >= -GOLD_DECLINE_LIMIT)
        evaluations.append({
            "run3_weight": float(weight), "promote": promote,
            "known_anchor_macro_auc": anchor_known,
            "known_candidate_macro_auc": candidate_known, "known_delta": known_delta,
            "bootstrap_delta_ci95_lower": lower, "bootstrap_samples": BOOTSTRAP_SAMPLES,
            "fold_wins": fold_wins, "fold_metrics": fold_metrics,
            "gold_anchor_macro_auc": anchor_gold, "gold_candidate_macro_auc": candidate_gold,
            "gold_delta": gold_delta, "gold_available": gold_available,
            "gold_decline_limit": GOLD_DECLINE_LIMIT,
        })
    passing = [item for item in evaluations if item["promote"]]
    selected = sorted(passing, key=lambda item: (-item["known_candidate_macro_auc"],
                                                  item["run3_weight"]))[0] if passing else None
    return {"promote": selected is not None, "selected_run3_weight":
            (selected["run3_weight"] if selected else 0.0),
            "anchor_known_macro_auc": anchor_known, "anchor_gold_macro_auc": anchor_gold,
            "candidates": evaluations, "bootstrap_samples": BOOTSTRAP_SAMPLES,
            "required_fold_wins": max(1, N_FOLDS - 1)}


def fold_mapping_fingerprint(frame, folds, groups):
    payload = [{"uid": str(uid), "fold": int(fold), "report_group_sha256": str(group)}
               for uid, fold, group in zip(frame[UID], folds, groups)]
    return canonical_json_hash(payload)


def weight_manifest(training_result, label_sha, blend_gate, selected_recipe,
                    run2_manifest_sha, run2_bundle_sha, fold_fingerprint):
    manifest = {
        "schema_version": 3, "recipe": "hierarchical-mil-run3-v1",
        "targets": TARGETS, "slots": [slot[0] for slot in SLOTS], "crop_mm": CROP_MM,
        "architecture": "target-patch-slot-window-mil-v1",
        "pool": POOL, "unfreeze_last": UNFREEZE_LAST, "n_folds": N_FOLDS,
        "label_sha256": label_sha, "selected_recipe": selected_recipe,
        "fold_mapping_fingerprint": fold_fingerprint,
        "run2_anchor": {
            "recipe": "detail_depth_70_30", "weights": list(RUN2_BLEND_WEIGHTS),
            "manifest_sha256": run2_manifest_sha, "bundle_sha256": run2_bundle_sha,
        },
        "run3_blend_candidates": list(RUN3_BLEND_CANDIDATES), "blend_gate": blend_gate,
        "bank": {
            "config": bank_config(HIERARCHICAL_BANK),
            "config_fingerprint": bank_fingerprint(HIERARCHICAL_BANK),
            "complete": bool(training_result["complete"]),
            "completed_folds": training_result["completed_folds"],
            "members": training_result["members"],
        },
    }
    manifest["manifest_fingerprint"] = canonical_json_hash(json_safe(manifest))
    atomic_json(MANIFEST_PATH, manifest)
    return manifest


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        number = float(value)
        return number if np.isfinite(number) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def atomic_json(path, value):
    temp = Path(str(path) + ".tmp")
    temp.write_text(json.dumps(json_safe(value), indent=2, sort_keys=True, allow_nan=False) + "\n")
    os.replace(temp, path)


def validate_manifest(manifest, run2_manifest_sha=None):
    if manifest.get("schema_version") != 3 or manifest.get("recipe") != "hierarchical-mil-run3-v1":
        raise PackageError("unsupported run-3 weight-manifest schema/recipe")
    if manifest.get("targets") != TARGETS or manifest.get("slots") != [slot[0] for slot in SLOTS]:
        raise PackageError("manifest target/slot order mismatch")
    valid_recipes = {"run2_anchor"} | {
        f"run2_anchor_run3_{int(round(weight * 100)):02d}" for weight in RUN3_BLEND_CANDIDATES}
    if manifest.get("selected_recipe") not in valid_recipes:
        raise PackageError("manifest selected_recipe is invalid")
    expected_manifest_fp = manifest.get("manifest_fingerprint")
    unhashed = dict(manifest)
    unhashed.pop("manifest_fingerprint", None)
    if expected_manifest_fp != canonical_json_hash(json_safe(unhashed)):
        raise PackageError("manifest fingerprint mismatch")
    stored = manifest.get("bank", {})
    if stored.get("config_fingerprint") != bank_fingerprint(HIERARCHICAL_BANK):
        raise PackageError("hierarchical bank configuration fingerprint mismatch")
    if stored.get("complete"):
        folds = sorted(int(member["fold"]) for member in stored.get("members", []))
        if folds != list(range(N_FOLDS)):
            raise PackageError("hierarchical package does not contain equal complete folds")
    if manifest["selected_recipe"] != "run2_anchor" and not stored.get("complete"):
        raise PackageError("selected run-3 blend refers to an incomplete hierarchical bank")
    anchor = manifest.get("run2_anchor", {})
    if anchor.get("recipe") != "detail_depth_70_30" or anchor.get("weights") != list(RUN2_BLEND_WEIGHTS):
        raise PackageError("run-2 anchor recipe mismatch")
    if run2_manifest_sha is not None and anchor.get("manifest_sha256") != run2_manifest_sha:
        raise PackageError("run-3 package refers to a different run-2 manifest")


def load_model_member(weights_dir, bank, member, device):
    activate_bank(bank)
    if member.get("config_fingerprint") != bank_fingerprint(bank):
        raise PackageError(f"{member.get('name')} bank fingerprint mismatch")
    path = weights_dir / member["file"]
    if not path.is_file() or sha256_file(path) != member.get("weight_sha256"):
        raise PackageError(f"{member.get('name')} weight file missing or SHA-256 mismatch")
    model = build_seeded_model(device, stable_seed(bank["seed"], member["fold"], "training"))
    state = torch.load(path, map_location="cpu")
    model.load_state_dict(state, strict=True)
    observed = np.asarray(hierarchical_model_fingerprint(model, device, bank), dtype=np.float64)
    expected = np.asarray(member.get("model_fingerprint", []), dtype=np.float64)
    if observed.shape != expected.shape or np.max(np.abs(observed - expected)) > FINGERPRINT_TOL:
        raise PackageError(f"{member.get('name')} model fingerprint mismatch")
    model.eval()
    return model


def infer_hierarchical_with_cache(bank, bank_manifest, cache_bundle, weights_dir):
    if not bank_manifest.get("complete"):
        raise PackageError(f"{bank['name']} bank is incomplete")
    predictions = []
    for index, member in enumerate(sorted(bank_manifest["members"], key=lambda item: int(item["fold"]))):
        device = DEVS[index % len(DEVS)]
        model = load_model_member(weights_dir, bank, member, device)
        predictions.append(predict_hierarchical_member(
            model, cache_bundle, device, tag=f"{bank['name']} fold{member['fold']} test "))
        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return equal_fold_rank_mean(predictions)


def load_run2_model_member(bundle, bank, member, device):
    activate_bank(bank)
    relative = f"weights/{member['file']}"
    if bundle.sha256(relative) != member.get("weight_sha256"):
        raise PackageError(f"run-2 {member.get('name')} SHA-256 mismatch")
    with _MODEL_INIT_LOCK:
        seed = stable_seed(bank["seed"], member["fold"], "training")
        torch.manual_seed(seed)
        if device.type == "cuda":
            torch.cuda.manual_seed_all(seed)
        model = _build_run2_model(device)
    model.load_state_dict(bundle.torch_state(relative), strict=True)
    observed = np.asarray(run2_model_fingerprint(model, device, bank), dtype=np.float64)
    expected = np.asarray(member.get("model_fingerprint", []), dtype=np.float64)
    if observed.shape != expected.shape or np.max(np.abs(observed - expected)) > FINGERPRINT_TOL:
        raise PackageError(f"run-2 {member.get('name')} model fingerprint mismatch")
    model.eval()
    return model


def infer_run2_with_cache(bank, bank_manifest, cache_bundle, bundle):
    predictions = []
    for index, member in enumerate(sorted(bank_manifest["members"], key=lambda item: int(item["fold"]))):
        device = DEVS[index % len(DEVS)]
        model = load_run2_model_member(bundle, bank, member, device)
        predictions.append(predict_member(
            model, cache_bundle, device, tag=f"run2 {bank['name']} fold{member['fold']} test "))
        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return equal_fold_rank_mean(predictions)


def submission_frame(uids, prediction):
    prediction = np.asarray(prediction, dtype=np.float64)
    if prediction.shape != (len(uids), N_TARGET):
        raise RuntimeError(f"prediction shape is {prediction.shape}, expected {(len(uids), N_TARGET)}")
    if not np.isfinite(prediction).all() or (prediction < 0).any() or (prediction > 1).any():
        raise RuntimeError("submission predictions must be finite and in [0,1]")
    frame = pd.DataFrame({UID: list(map(str, uids))})
    for index, target in enumerate(TARGETS):
        frame[target] = prediction[:, index]
    expected = [UID] + TARGETS
    sample_path = ROOT / "sample_submission.csv"
    if sample_path.is_file():
        sample = pd.read_csv(sample_path, dtype={UID: str})
        if list(sample.columns) != expected:
            raise RuntimeError("sample_submission.csv schema does not match the 12-target contract")
        if list(sample[UID]) != list(frame[UID]):
            raise RuntimeError("submission IDs/order differ from sample_submission.csv")
    if list(frame.columns) != expected or not frame[UID].is_unique:
        raise RuntimeError("submission schema or ID uniqueness failed")
    constants = [target for target in TARGETS if frame[target].nunique() <= 1]
    if constants:
        if len(frame) > VISIBLE_TEST_STUB_MAX_ROWS:
            raise RuntimeError(f"constant target columns are not auditable model output: {constants}")
        log(f"submission audit warning: constant targets {constants} on Kaggle's "
            f"{len(frame)}-row visible test stub; the non-constant guard remains mandatory "
            "for every larger evaluation set")
    return frame


def write_submission_artifact(uids, prediction, path):
    frame = submission_frame(uids, prediction)
    constant_targets = [target for target in TARGETS if frame[target].nunique() <= 1]
    stub_exemption = bool(constant_targets and len(frame) <= VISIBLE_TEST_STUB_MAX_ROWS)
    temp = Path(str(path) + ".tmp")
    frame.to_csv(temp, index=False)
    os.replace(temp, path)
    return {"path": str(path), "sha256": sha256_file(path), "rows": len(frame),
            "columns": len(frame.columns),
            "constant_targets": constant_targets,
            "visible_test_stub_exemption": stub_exemption,
            "ranges": {target: [float(frame[target].min()), float(frame[target].max())]
                       for target in TARGETS}}


def submission_schema_checks(artifacts):
    selected = artifacts["selected"]
    constants = list(selected.get("constant_targets", []))
    stub_exemption = bool(selected.get("visible_test_stub_exemption", False))
    return {
        "target_order": TARGETS,
        "test_ids_in_sample_order": True,
        "finite_in_range": True,
        "nonconstant": not constants,
        "constant_targets": constants,
        "visible_test_stub_exemption": stub_exemption,
        # Kept for receipt compatibility, but remains truthful: it is false when the
        # three-row stub ties. submission_audit_passed records the narrow exception.
        "finite_nonconstant_in_range": not constants,
        "submission_audit_passed": not constants or stub_exemption,
    }


def prediction_metrics(supervision, prediction):
    known = per_target_auc(supervision["teacher_y"], prediction, supervision["known_mask"])
    all_teacher = per_target_auc(supervision["teacher_y"], prediction)
    gold = per_target_auc(supervision["gold_y"], prediction,
                          supervision["gold_mask"], min_pos=1)
    return {"known_teacher_per_target_auc": known,
            "known_teacher_macro_auc": macro_from_metrics(known),
            "all_teacher_per_target_auc": all_teacher,
            "all_teacher_macro_auc": macro_from_metrics(all_teacher),
            "gold_per_target_auc": gold, "gold_macro_auc": macro_from_metrics(gold)}


def save_oof(frame, folds, groups, supervision, run2_oof, run3_result, blend_gate):
    output = pd.DataFrame({UID: frame[UID].astype(str), "report_group_sha256": groups, "fold": folds})
    run3_prediction = (run3_result["oof"] if run3_result.get("complete")
                       else np.full_like(run2_oof["anchor"], np.nan))
    for index, target in enumerate(TARGETS):
        output[f"teacher__{target}"] = supervision["teacher_y"][:, index]
        output[f"teacher_known__{target}"] = supervision["known_mask"][:, index]
        output[f"gold__{target}"] = supervision["gold_y"][:, index]
        output[f"run2_detail__{target}"] = run2_oof["detail"][:, index]
        output[f"run2_depth__{target}"] = run2_oof["depth"][:, index]
        output[f"run2_anchor__{target}"] = run2_oof["anchor"][:, index]
        output[f"run3_hierarchical__{target}"] = run3_prediction[:, index]
    output.to_csv(OOF_PATH, index=False)
    metrics = {
        "run2_anchor": {
            "complete": True,
            "legacy_receipt_known_teacher_macro_auc": run2_oof["legacy_known_macro_auc"],
            **prediction_metrics(supervision, run2_oof["anchor"]),
        },
        "run3_hierarchical": ({"complete": True, **prediction_metrics(supervision, run3_prediction)}
                              if run3_result.get("complete") else {"complete": False}),
        "blend_gate": blend_gate,
        "supervision_counts": {
            target: {"known": int(supervision["known_mask"][:, index].sum()),
                     "unknown": int((~supervision["known_mask"][:, index]).sum()),
                     "gold": int(supervision["gold_mask"][:, index].sum())}
            for index, target in enumerate(TARGETS)},
    }
    atomic_json(METRICS_PATH, metrics)
    return metrics


def run_inference(manifest, test_meta, weights_dir, run2_bundle, run2_manifest):
    run2_manifest_sha = run2_bundle.sha256("weights/manifest.json")
    validate_manifest(manifest, run2_manifest_sha)
    artifacts = {}
    detail_cache = build_exact_cache(
        test_meta, RUN2_BANKS[0], "run2-detail-test", deadline=T0 + TIME_BUDGET)
    try:
        detail = infer_run2_with_cache(
            RUN2_BANKS[0], run2_manifest["banks"]["detail"], detail_cache, run2_bundle)
    finally:
        release_cache(detail_cache["pixels"], detail_cache["allocation"])

    depth_cache = build_exact_cache(
        test_meta, RUN2_BANKS[1], "shared-depth-test", deadline=T0 + TIME_BUDGET)
    try:
        depth = infer_run2_with_cache(
            RUN2_BANKS[1], run2_manifest["banks"]["depth"], depth_cache, run2_bundle)
        anchor = RUN2_BLEND_WEIGHTS[0] * detail + RUN2_BLEND_WEIGHTS[1] * depth
        artifacts["run2_anchor"] = write_submission_artifact(
            TEST_DF[UID], anchor, ANCHOR_SUBMISSION_PATH)
        # The verified 0.877 recipe is banked before any optional run-3 model is loaded.
        artifacts["selected"] = write_submission_artifact(TEST_DF[UID], anchor, SUBMISSION_PATH)

        if manifest["bank"].get("complete"):
            run3 = infer_hierarchical_with_cache(
                HIERARCHICAL_BANK, manifest["bank"], depth_cache, weights_dir)
            artifacts["run3_only"] = write_submission_artifact(
                TEST_DF[UID], run3, RUN3_SUBMISSION_PATH)
            anchor_rank, run3_rank = percentile_rank(anchor), percentile_rank(run3)
            candidate_predictions = {}
            for weight in RUN3_BLEND_CANDIDATES:
                prediction = (1.0 - weight) * anchor_rank + weight * run3_rank
                key = f"anchor_run3_{int(round(weight * 100)):02d}"
                candidate_predictions[weight] = prediction
                artifacts[key] = write_submission_artifact(
                    TEST_DF[UID], prediction, CANDIDATE_SUBMISSION_PATHS[weight])
            selected_weight = float(manifest["blend_gate"].get("selected_run3_weight", 0.0))
            if manifest["selected_recipe"] != "run2_anchor":
                if selected_weight not in candidate_predictions:
                    raise PackageError("selected run-3 blend weight is absent from the candidate grid")
                artifacts["selected"] = write_submission_artifact(
                    TEST_DF[UID], candidate_predictions[selected_weight], SUBMISSION_PATH)
        else:
            artifacts["run3_only"] = {"available": False, "reason": "incomplete bank discarded"}
    finally:
        release_cache(depth_cache["pixels"], depth_cache["allocation"])
    return artifacts


def training_main():
    run2_bundle = resolve_run2_bundle()
    run2_manifest, run2_receipt = verify_run2_bundle(run2_bundle)
    supervision_full = load_supervision(TRAIN_DF)
    if supervision_full["label_sha256"] != run2_manifest["label_sha256"]:
        raise PackageError("active label table is not the label table used by run 2")
    run2_full = load_run2_oof(run2_bundle, TRAIN_DF, supervision_full)
    train_uids = smoke_train_uids(TRAIN_DF, supervision_full)
    row_lookup = {uid: index for index, uid in enumerate(TRAIN_DF[UID].astype(str))}
    rows = np.asarray([row_lookup[uid] for uid in train_uids], dtype=int)
    frame = TRAIN_DF.iloc[rows].reset_index(drop=True)
    supervision = {key: (value[rows] if isinstance(value, np.ndarray) and len(value) == len(TRAIN_DF) else value)
                   for key, value in supervision_full.items()}
    folds = run2_full["folds"][rows].astype(np.int16)
    if MODE == "smoke":
        folds = (folds % N_FOLDS).astype(np.int16)
    groups = run2_full["groups"][rows]
    if sorted(set(folds)) != list(range(N_FOLDS)):
        raise PackageError("active subset does not contain every required run-2 fold")
    run2_oof = {key: (value[rows] if isinstance(value, np.ndarray) and len(value) == len(TRAIN_DF) else value)
                for key, value in run2_full.items() if key != "frame"}
    metadata = prepare_shared_metadata(train_uids)
    inference_start = T0 + TIME_BUDGET - INFERENCE_RESERVE
    if time.time() < inference_start:
        run3_result = train_bank(
            HIERARCHICAL_BANK, metadata["train"], supervision, folds, inference_start,
            run2_bundle, run2_manifest)
    else:
        run3_result = {"name": "hierarchical", "complete": False, "members": [],
                       "completed_folds": [], "errors": {"deadline": "inference reserve reached"}}
    if run3_result["complete"]:
        blend_gate = evaluate_run3_blend_gate(
            run2_oof["anchor"], run3_result["oof"], supervision, groups, folds)
    else:
        log("hierarchical bank incomplete: it is discarded and cannot vote")
        blend_gate = {"promote": False, "selected_run3_weight": 0.0,
                      "reason": "hierarchical bank incomplete", "bootstrap_samples": BOOTSTRAP_SAMPLES}
    selected_weight = float(blend_gate.get("selected_run3_weight", 0.0))
    selected = (f"run2_anchor_run3_{int(round(selected_weight * 100)):02d}"
                if blend_gate.get("promote") else "run2_anchor")
    metrics = save_oof(frame, folds, groups, supervision, run2_oof, run3_result, blend_gate)
    run2_manifest_sha = run2_bundle.sha256("weights/manifest.json")
    manifest = weight_manifest(
        run3_result, supervision["label_sha256"], blend_gate, selected,
        run2_manifest_sha, run2_bundle.bundle_sha256,
        fold_mapping_fingerprint(frame, folds, groups))
    metadata.pop("train", None)
    gc.collect()
    artifacts = run_inference(manifest, metadata["test"], OUT_WEIGHTS, run2_bundle, run2_manifest)
    receipt = {
        "schema_version": 3, "mode": MODE, "recipe": manifest["recipe"],
        "selected_recipe": selected, "production_submission": MODE == "train",
        "configuration_fingerprints": {"hierarchical": bank_fingerprint(HIERARCHICAL_BANK),
                                        "fold_mapping": manifest["fold_mapping_fingerprint"]},
        "label_sha256": supervision["label_sha256"],
        "run2_bundle_sha256": run2_bundle.bundle_sha256,
        "run2_manifest_sha256": run2_manifest_sha,
        "run2_selected_recipe": run2_receipt.get("selected_recipe"),
        "fold_completion": run3_result["completed_folds"],
        "blend_decision": blend_gate, "runtime_seconds": time.time() - T0,
        "schema_checks": submission_schema_checks(artifacts),
        "artifacts": artifacts, "oof_metrics": metrics,
        "manifest_sha256": sha256_file(MANIFEST_PATH),
    }
    atomic_json(RECEIPT_PATH, receipt)
    return receipt


def inference_main():
    run2_bundle = resolve_run2_bundle()
    run2_manifest, run2_receipt = verify_run2_bundle(run2_bundle)
    explicit = os.environ.get("RSNA_WEIGHTS_DIR", "").strip()
    if not explicit:
        raise PackageError("RSNA_WEIGHTS_DIR is required in infer mode")
    weights_dir = Path(explicit)
    manifest_path = weights_dir / "manifest.json"
    if not manifest_path.is_file():
        raise PackageError(f"weights manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text())
    validate_manifest(manifest, run2_bundle.sha256("weights/manifest.json"))
    metadata = prepare_shared_metadata(None)
    artifacts = run_inference(manifest, metadata["test"], weights_dir, run2_bundle, run2_manifest)
    receipt = {
        "schema_version": 3, "mode": MODE, "recipe": manifest["recipe"],
        "selected_recipe": manifest["selected_recipe"], "production_submission": True,
        "configuration_fingerprints": {"hierarchical": bank_fingerprint(HIERARCHICAL_BANK),
                                        "fold_mapping": manifest["fold_mapping_fingerprint"]},
        "label_sha256": manifest["label_sha256"],
        "run2_bundle_sha256": run2_bundle.bundle_sha256,
        "run2_manifest_sha256": run2_bundle.sha256("weights/manifest.json"),
        "run2_selected_recipe": run2_receipt.get("selected_recipe"),
        "fold_completion": manifest["bank"]["completed_folds"],
        "blend_decision": manifest["blend_gate"], "runtime_seconds": time.time() - T0,
        "schema_checks": submission_schema_checks(artifacts),
        "artifacts": artifacts, "source_manifest_sha256": sha256_file(manifest_path),
    }
    atomic_json(RECEIPT_PATH, receipt)
    return receipt


def main():
    if MODE == "infer":
        receipt = inference_main()
    else:
        receipt = training_main()
    log(f"completed {MODE} in {receipt['runtime_seconds'] / 60:.1f} min; "
        f"selected={receipt['selected_recipe']}; receipt={RECEIPT_PATH}")
    return receipt


In [ ]:
# Run: production failures are intentionally not converted to fallback submissions.
RUN_RECEIPT = main()


## Output and acceptance checklist

A successful training run writes:

- `submission.csv`, `submission_run2_anchor.csv`, `submission_run3_only.csv`, and every
  predeclared candidate blend;
- `oof_predictions.csv` and `oof_metrics.json`;
- `weights/manifest.json` with per-file, model, and configuration fingerprints;
- `run_receipt.json` with run-2 bundle lineage, label/fold fingerprints, fold completion,
  gate decision, runtime, schema checks, prediction ranges, and submission SHA-256.

An incomplete hierarchical bank is deleted and cannot vote. Final inference owns the last
150 minutes of the eight-hour budget. A predeclared 15%, 30%, or 45% run-3 vote is promoted
only when explicit-label macro AUC improves, the 2,000-sample report-group bootstrap lower
95% bound is non-negative, at least four folds improve, and gold macro AUC falls by no more
than 0.01. Otherwise the verified run-2 anchor is submitted.

The only leaderboard acceptance criterion is an official Kaggle score above **0.877**.
OOF metrics, filenames, titles, and historical notebook receipts are not a substitute.
